# Notebook 04 — Validasi Statistik Lanjutan & Justifikasi Threshold

**Skripsi: Sistem Prediksi Dini Diabetes (DiaPredict) — Revisi Pengujian V3**

---

## Latar belakang revisi

Pada notebook versi sebelumnya (V2), seluruh klaim performa model bertumpu pada **satu kali
holdout 80:20** ditambah 5-fold cross validation yang hanya dipakai pada tahap tuning. Skema ini
memiliki tiga kelemahan metodologis yang wajar dipersoalkan penguji:

1. **Satu titik estimasi tanpa ukuran ketidakpastian.** Angka recall 0.9057 untuk Random Forest
   adalah hasil dari satu partisi data tertentu. Bila partisi diganti (random_state berbeda),
   angka tersebut akan bergerak. Tanpa simpangan baku dan interval kepercayaan, pembaca tidak
   tahu apakah selisih antar model bersifat nyata atau sekadar variasi acak pengambilan sampel.
2. **Estimasi optimistik akibat tuning.** Hyperparameter dipilih memakai skor cross validation,
   lalu skor cross validation yang sama dilaporkan sebagai performa model. Praktik ini
   menimbulkan *optimistic bias* karena data validasi ikut "dilihat" saat pemilihan model.
3. **Threshold 0.4965 belum dijustifikasi secara formal.** Nilai tersebut dipakai di website
   produksi, namun belum ada perbandingan tertulis dengan strategi penentuan threshold lain
   sehingga terkesan angka ajaib.

## Yang ditambahkan pada notebook ini

| No | Eksperimen | Menjawab |
|----|------------|----------|
| 1 | **Repeated Stratified K-Fold CV** (5 fold x 5 repetisi = 25 estimasi per model) | ketidakpastian, interval kepercayaan 95% |
| 2 | **Nested Cross Validation** (outer 5 fold, inner 3 fold) | estimasi performa tak bias, besar bias optimistik tuning |
| 3 | **Empat uji statistik**: McNemar, 5x2cv paired t-test (Dietterich), Wilcoxon signed-rank, DeLong | apakah selisih antar model signifikan secara statistik |
| 4 | **Delapan strategi penentuan threshold** | justifikasi formal "kenapa 0.4965" |
| 5 | **Analisis kalibrasi probabilitas + Decision Curve Analysis** | apakah angka probabilitas layak ditampilkan ke pengguna, dan apakah model berguna secara klinis |
| 6 | **Validation curve hyperparameter** | sensitivitas model terhadap hyperparameter, deteksi overfitting |

Seluruh hasil disimpan sebagai tabel CSV, gambar PNG, dan satu berkas JSON
`hasil_validasi_statistik.json` yang akan digabungkan oleh notebook `06`.

**Baseline pembanding (hasil notebook V2, 1x holdout 80:20):**

| Model | Recall | ROC-AUC | Threshold |
|---|---|---|---|
| Random Forest | 0.9057 | 0.9733 | 0.4965 |
| KNN | 0.9121 | 0.9524 | 0.3810 |
| SVM (Linear) | 0.8833 | 0.9581 | 0.4951 |

Threshold yang dipakai website produksi saat ini: **0.4965**.


In [ ]:
# ============================================================
# CELL 1: Instalasi Library
# ============================================================
!pip install -q pandas numpy matplotlib seaborn scikit-learn imbalanced-learn statsmodels kagglehub

In [ ]:
# ============================================================
# CELL 2: Import & Konstanta Global
# ============================================================
import os, json, time, math, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, StratifiedShuffleSplit,
    RepeatedStratifiedKFold, cross_validate, cross_val_predict,
    learning_curve, validation_curve, GridSearchCV, RandomizedSearchCV
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, average_precision_score,
    precision_recall_curve, confusion_matrix, classification_report,
    brier_score_loss
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

SELECTED_FEATURES = ['age', 'bmi', 'hypertension', 'HbA1c_level', 'blood_glucose_level']
FEATURE_LABELS    = ['Usia', 'BMI', 'Hipertensi', 'HbA1c', 'Kadar Glukosa']
TARGET            = 'diabetes'

WARNA_MODEL = {'Random Forest': '#3498db', 'KNN': '#e74c3c', 'SVM (Linear)': '#2ecc71'}
WARNA_AKSEN = '#f39c12'

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25
sns.set_style('whitegrid')

# --- Folder output -------------------------------------------------------
# Set PAKAI_DRIVE = True bila ingin hasil tersimpan permanen di Google Drive
# (WAJIB True kalau ingin notebook 06 membaca hasil notebook 01-05).
PAKAI_DRIVE = False

if PAKAI_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/DiaPredict_Revisi'
else:
    OUTPUT_DIR = '/content/hasil_revisi'

for sub in ['', '/tabel', '/gambar', '/json']:
    os.makedirs(OUTPUT_DIR + sub, exist_ok=True)

print(f'Folder output : {OUTPUT_DIR}')
print(f'Fitur         : {SELECTED_FEATURES}')

In [ ]:
# ============================================================
# CELL 3: Fungsi Utilitas Penyimpanan Hasil
# ============================================================
def simpan_tabel(df, nama, tampilkan=True):
    """Simpan DataFrame ke CSV di OUTPUT_DIR/tabel dan tampilkan."""
    path = f'{OUTPUT_DIR}/tabel/{nama}.csv'
    df.to_csv(path, index=False)
    print(f'[TABEL DISIMPAN] {path}')
    if tampilkan:
        display(df)
    return df

def simpan_json(obj, nama):
    """Simpan dict/list hasil eksperimen ke JSON (dipakai notebook 06 & website)."""
    path = f'{OUTPUT_DIR}/json/{nama}.json'
    def _konversi(o):
        if isinstance(o, (np.integer,)):  return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, (np.ndarray,)):  return o.tolist()
        if isinstance(o, (np.bool_,)):    return bool(o)
        return str(o)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=_konversi)
    print(f'[JSON DISIMPAN] {path}')
    return obj

def simpan_gambar(nama, fig=None, dpi=150):
    """Simpan figure matplotlib aktif ke OUTPUT_DIR/gambar."""
    path = f'{OUTPUT_DIR}/gambar/{nama}.png'
    (fig or plt).savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'[GAMBAR DISIMPAN] {path}')
    return path

def garis(judul='', lebar=70):
    print('=' * lebar)
    if judul:
        print(f'  {judul}')
        print('=' * lebar)

In [ ]:
# ============================================================
# CELL 4: Load Dataset + Cleaning + Winsorization
# (Identik dengan pipeline notebook V2 agar hasil dapat dibandingkan)
# ============================================================
import kagglehub

def muat_dan_bersihkan_data(verbose=True):
    path = kagglehub.dataset_download('iammustafatz/diabetes-prediction-dataset')
    csv_file = os.path.join(path, 'diabetes_prediction_dataset.csv')
    df_raw = pd.read_csv(csv_file)

    # 1) Hapus duplikat pada dataset penuh (SAMA seperti V2 -> sisa 96.146 baris)
    df = df_raw.drop_duplicates().reset_index(drop=True)

    # 2) Ambil 5 fitur terpilih + target
    df = df[SELECTED_FEATURES + [TARGET]].copy()

    # 3) Winsorization (capping IQR) hanya untuk fitur numerik non-biner
    numeric_feats = [f for f in SELECTED_FEATURES if df[f].nunique() > 2]
    ringkas = []
    for feat in numeric_feats:
        Q1, Q3 = df[feat].quantile(0.25), df[feat].quantile(0.75)
        IQR = Q3 - Q1
        low, up = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        n_cap = int(((df[feat] < low) | (df[feat] > up)).sum())
        df[feat] = df[feat].clip(lower=low, upper=up)
        ringkas.append({'fitur': feat, 'batas_bawah': low, 'batas_atas': up, 'n_dicapping': n_cap})

    if verbose:
        garis('DATA SIAP PAKAI')
        print(f'Baris (setelah hapus duplikat) : {len(df):,}')
        print(f'Distribusi kelas               : '
              f'{(df[TARGET]==0).sum():,} sehat / {(df[TARGET]==1).sum():,} diabetes '
              f'({df[TARGET].mean()*100:.2f}% positif)')
        display(pd.DataFrame(ringkas))
    return df

df_clean = muat_dan_bersihkan_data()
X_all = df_clean[SELECTED_FEATURES].copy()
y_all = df_clean[TARGET].copy()

In [ ]:
# ============================================================
# CELL 5: Pabrik Pipeline Model (anti data leakage)
# Urutan: StandardScaler -> SMOTE -> Classifier (imblearn Pipeline,
# sehingga SMOTE HANYA aktif saat fit, tidak saat predict/validasi)
# ============================================================

# Hyperparameter terbaik hasil tuning notebook V2 (baseline pembanding)
PARAM_RF_V2  = dict(n_estimators=200, max_depth=10, min_samples_split=5,
                    min_samples_leaf=4, max_features='log2', criterion='entropy',
                    class_weight='balanced')
PARAM_KNN_V2 = dict(n_neighbors=21, weights='uniform', metric='euclidean', leaf_size=20)
PARAM_SVM_V2 = dict(C=0.1, max_iter=3000)

def buat_pipeline_rf(pakai_smote=True, **params):
    p = {**PARAM_RF_V2, **params}
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, **p)))
    return ImbPipeline(langkah)

def buat_pipeline_knn(pakai_smote=True, **params):
    p = {**PARAM_KNN_V2, **params}
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', KNeighborsClassifier(n_jobs=-1, **p)))
    return ImbPipeline(langkah)

def buat_pipeline_svm(pakai_smote=True, kernel='linear', C=0.1, gamma='scale',
                      degree=3, max_iter=3000, kalibrasi='sigmoid'):
    """kernel='linear' -> LinearSVC (cepat). Kernel lain -> SVC."""
    if kernel == 'linear':
        base = LinearSVC(C=C, max_iter=max_iter, class_weight='balanced',
                         dual=False, random_state=RANDOM_STATE)
    else:
        base = SVC(kernel=kernel, C=C, gamma=gamma, degree=degree,
                   class_weight='balanced', random_state=RANDOM_STATE)
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', CalibratedClassifierCV(base, cv=3, method=kalibrasi)))
    return ImbPipeline(langkah)

PABRIK_MODEL = {
    'Random Forest': buat_pipeline_rf,
    'KNN'          : buat_pipeline_knn,
    'SVM (Linear)' : buat_pipeline_svm,
}

In [ ]:
# ============================================================
# CELL 6: Fungsi Evaluasi Standar (dipakai seluruh notebook)
# ============================================================
def threshold_youden(y_true, y_proba):
    fpr, tpr, thr = roc_curve(y_true, y_proba)
    return float(thr[np.argmax(tpr - fpr)])

def hitung_metrik(y_true, y_pred, y_proba=None):
    hasil = {
        'accuracy' : accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall'   : recall_score(y_true, y_pred, zero_division=0),
        'f1'       : f1_score(y_true, y_pred, zero_division=0),
    }
    if y_proba is not None:
        hasil['roc_auc']  = roc_auc_score(y_true, y_proba)
        hasil['ap_score'] = average_precision_score(y_true, y_proba)
        hasil['brier']    = brier_score_loss(y_true, y_proba)
    return hasil

def evaluasi_holdout(model, X_tr, y_tr, X_te, y_te, tuning_threshold=True):
    """Fit -> prediksi -> metrik pada threshold 0.5 dan threshold Youden."""
    t0 = time.time(); model.fit(X_tr, y_tr); waktu_latih = time.time() - t0
    t0 = time.time(); y_proba = model.predict_proba(X_te)[:, 1]; waktu_infer = time.time() - t0

    thr = threshold_youden(y_te, y_proba) if tuning_threshold else 0.5
    m_def  = hitung_metrik(y_te, (y_proba >= 0.5).astype(int), y_proba)
    m_tune = hitung_metrik(y_te, (y_proba >= thr).astype(int), y_proba)
    return {
        'threshold': thr,
        'waktu_latih_s': waktu_latih,
        'waktu_infer_ms': waktu_infer * 1000,
        **{f'{k}_default': v for k, v in m_def.items()},
        **{f'{k}_tuned'  : v for k, v in m_tune.items()},
    }

def ci95_proporsi(p, n):
    """Confidence interval 95% (Wald) untuk metrik berbasis proporsi (mis. recall)."""
    if n == 0: return (np.nan, np.nan, np.nan)
    se = math.sqrt(max(p * (1 - p), 1e-12) / n)
    return (p - 1.96 * se, p + 1.96 * se, 1.96 * se)

In [ ]:
# ============================================================
# CELL 7: Konstanta Eksperimen, Subsample, dan Split Data
# ============================================================
from scipy import stats as sstats
from scipy.stats import wilcoxon, norm
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.calibration import calibration_curve

# MODE_CEPAT = True  -> subsample stratified N_SUBSAMPLE baris (uji coba / Colab CPU)
# MODE_CEPAT = False -> data penuh 96.146 baris (dipakai untuk angka final skripsi)
MODE_CEPAT  = True
N_SUBSAMPLE = 30000

def ambil_subsample(X, y, n, seed=RANDOM_STATE):
    """Subsample stratified: proporsi kelas positif dipertahankan."""
    if n >= len(X):
        return X, y
    sss = StratifiedShuffleSplit(n_splits=1, train_size=n, random_state=seed)
    idx, _ = next(sss.split(X, y))
    return X.iloc[idx], y.iloc[idx]

if MODE_CEPAT:
    X_eks, y_eks = ambil_subsample(X_all, y_all, N_SUBSAMPLE)
else:
    X_eks, y_eks = X_all.copy(), y_all.copy()

X_eks = X_eks.reset_index(drop=True)
y_eks = y_eks.reset_index(drop=True)

# Holdout 80:20 stratified (skema yang sama dengan notebook V2)
X_train, X_test, y_train, y_test = train_test_split(
    X_eks, y_eks, test_size=0.20, stratify=y_eks, random_state=RANDOM_STATE)
X_train = X_train.reset_index(drop=True); y_train = y_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True);  y_test  = y_test.reset_index(drop=True)
y_test_np = np.asarray(y_test).astype(int)

# Ukuran data untuk eksperimen berat (dibatasi supaya sesi Colab tidak time-out)
N_NESTED = min(12000, len(X_eks)) if MODE_CEPAT else len(X_eks)
N_5X2CV  = min(10000, len(X_eks)) if MODE_CEPAT else min(30000, len(X_eks))
N_VC     = min(12000, len(X_eks)) if MODE_CEPAT else min(30000, len(X_eks))

X_nest, y_nest = ambil_subsample(X_eks, y_eks, N_NESTED)
X_52,   y_52   = ambil_subsample(X_eks, y_eks, N_5X2CV)
X_vc,   y_vc   = ambil_subsample(X_eks, y_eks, N_VC)
X_nest = X_nest.reset_index(drop=True); y_nest = y_nest.reset_index(drop=True)
X_52   = X_52.reset_index(drop=True);   y_52   = y_52.reset_index(drop=True)
X_vc   = X_vc.reset_index(drop=True);   y_vc   = y_vc.reset_index(drop=True)

# Baseline notebook V2 (1x holdout 80:20) sebagai pembanding di setiap eksperimen
BASELINE_V2 = {
    'Random Forest': {'recall': 0.9057, 'roc_auc': 0.9733, 'threshold': 0.4965},
    'KNN'          : {'recall': 0.9121, 'roc_auc': 0.9524, 'threshold': 0.3810},
    'SVM (Linear)' : {'recall': 0.8833, 'roc_auc': 0.9581, 'threshold': 0.4951},
}
THRESHOLD_PRODUKSI = 0.4965   # nilai yang dipakai website DiaPredict saat ini

garis('KONFIGURASI EKSPERIMEN NOTEBOOK 04')
print(f'MODE_CEPAT             : {MODE_CEPAT}')
print(f'Data eksperimen        : {len(X_eks):,} baris '
      f'({int(y_eks.sum()):,} positif / {y_eks.mean()*100:.2f}%)')
print(f'Train / Test (80:20)   : {len(X_train):,} / {len(X_test):,}')
print(f'Data nested CV         : {len(X_nest):,} baris')
print(f'Data 5x2cv t-test      : {len(X_52):,} baris')
print(f'Data validation curve  : {len(X_vc):,} baris')
print(f'Threshold produksi     : {THRESHOLD_PRODUKSI}')
print()
garis('ESTIMASI WAKTU EKSEKUSI (Colab CPU standar)')
print('Eksperimen 1 - Repeated CV 5x5 (3 model)   : ~4 - 8 menit')
print('Eksperimen 2 - Nested CV outer 5 x inner 3 : ~8 - 15 menit  (paling lama)')
print('Eksperimen 3 - Empat uji statistik          : ~3 - 6 menit')
print('Eksperimen 4 - Strategi threshold           : < 1 menit')
print('Eksperimen 5 - Kalibrasi + decision curve   : < 1 menit')
print('Eksperimen 6 - Validation curve             : ~4 - 8 menit')
print('TOTAL perkiraan (MODE_CEPAT=True)           : ~20 - 40 menit')
print('Dengan MODE_CEPAT=False waktu naik sekitar 3-4 kali lipat.')
print()
print('Catatan: setiap eksperimen menyimpan hasilnya sendiri (checkpoint),')
print('sehingga bila sesi Colab terputus tidak semua hasil hilang.')

---

# EKSPERIMEN 1 — Repeated Stratified K-Fold Cross Validation

**Masalah yang dijawab:** notebook V2 melaporkan satu angka recall dari satu partisi.
Angka itu tidak menyertakan informasi seberapa besar angka tersebut bisa berubah bila
partisi diganti.

**Desain:** `RepeatedStratifiedKFold(n_splits=5, n_repeats=5)` menghasilkan
**25 estimasi independen** per model (5 fold x 5 pengulangan dengan pengacakan berbeda).
Dari 25 nilai tersebut dihitung rata-rata, simpangan baku, dan **interval kepercayaan 95%**
memakai distribusi t:

$$\text{CI}_{95\%} = \bar{x} \pm t_{0{,}975;\,n-1}\cdot\frac{s}{\sqrt{n}},\qquad n = 25,\ df = 24$$

Skor latih (`return_train_score=True`) juga direkam untuk mendeteksi *overfitting*
(selisih train - validation yang besar menandakan model menghafal data latih).


In [ ]:
# ============================================================
# CELL 8: EKSPERIMEN 1 - Repeated Stratified K-Fold CV (5 fold x 5 repetisi)
# ============================================================
SCORING_CV = {
    'recall'            : 'recall',
    'precision'         : 'precision',
    'f1'                : 'f1',
    'roc_auc'           : 'roc_auc',
    'average_precision' : 'average_precision',
}
METRIK_CV = list(SCORING_CV.keys())

rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)

garis('EKSPERIMEN 1: REPEATED STRATIFIED K-FOLD CV (5 x 5 = 25 estimasi/model)')
print(f'Jumlah data  : {len(X_eks):,} baris')
print(f'Jumlah fit   : 25 fit x 3 model = 75 fit')
print('Estimasi     : sekitar 4 - 8 menit. Mohon tunggu, progres dicetak per model.')
print()

hasil_repeated = {}
t_mulai_e1 = time.time()

for nama, pabrik in PABRIK_MODEL.items():
    t0 = time.time()
    print(f'[MULAI] {nama} ...')
    res = cross_validate(
        pabrik(), X_eks, y_eks,
        cv=rskf, scoring=SCORING_CV,
        return_train_score=True, n_jobs=-1, error_score='raise'
    )
    hasil_repeated[nama] = res
    durasi = time.time() - t0
    print(f'[SELESAI] {nama:14s} | recall CV = {res["test_recall"].mean():.4f} '
          f'(sd {res["test_recall"].std(ddof=1):.4f}) | waktu {durasi/60:.2f} menit')

print()
print(f'Total waktu Eksperimen 1: {(time.time() - t_mulai_e1)/60:.2f} menit')

In [ ]:
# ============================================================
# CELL 9: Ringkasan Statistik + Interval Kepercayaan 95% (distribusi t)
# ============================================================
def ci95_t(nilai):
    """
    Interval kepercayaan 95% untuk rata-rata sampel kecil memakai distribusi t.
        CI = mean +/- t(0.975, df=n-1) * s / sqrt(n)
    Dipakai karena n = 25 (bukan sampel besar) dan simpangan baku populasi tidak diketahui.
    """
    v  = np.asarray(nilai, dtype=float)
    n  = len(v)
    m  = float(v.mean())
    s  = float(v.std(ddof=1))
    t_kritis = float(sstats.t.ppf(0.975, df=n - 1))
    margin   = t_kritis * s / math.sqrt(n)
    return {'mean': m, 'std': s, 'n': n, 't_kritis': t_kritis,
            'margin': margin, 'ci_bawah': m - margin, 'ci_atas': m + margin}

baris_rcv = []
for nama, res in hasil_repeated.items():
    for met in METRIK_CV:
        c   = ci95_t(res[f'test_{met}'])
        ctr = ci95_t(res[f'train_{met}'])
        baris_rcv.append({
            'model'        : nama,
            'metrik'       : met,
            'mean_val'     : round(c['mean'], 4),
            'std_val'      : round(c['std'], 4),
            'ci95_bawah'   : round(c['ci_bawah'], 4),
            'ci95_atas'    : round(c['ci_atas'], 4),
            'margin_error' : round(c['margin'], 4),
            'min_val'      : round(float(np.min(res[f'test_{met}'])), 4),
            'maks_val'     : round(float(np.max(res[f'test_{met}'])), 4),
            'mean_train'   : round(ctr['mean'], 4),
            'gap_train_val': round(ctr['mean'] - c['mean'], 4),
            'pelaporan'    : f"{c['mean']:.4f} +/- {c['margin']:.4f}",
        })

tabel_repeated_cv = pd.DataFrame(baris_rcv)
simpan_tabel(tabel_repeated_cv, 'tabel_repeated_cv')

garis('KESIMPULAN EKSPERIMEN 1 (siap salin ke skripsi)')
for nama in PABRIK_MODEL:
    r = tabel_repeated_cv[(tabel_repeated_cv['model'] == nama) &
                          (tabel_repeated_cv['metrik'] == 'recall')].iloc[0]
    a = tabel_repeated_cv[(tabel_repeated_cv['model'] == nama) &
                          (tabel_repeated_cv['metrik'] == 'roc_auc')].iloc[0]
    b = BASELINE_V2[nama]
    di_dalam = 'YA' if r['ci95_bawah'] <= b['recall'] <= r['ci95_atas'] else 'TIDAK'
    print(f'{nama}')
    print(f'  Recall  25 fold : {r["mean_val"]:.4f} +/- {r["margin_error"]:.4f} '
          f'(CI 95%: {r["ci95_bawah"]:.4f} - {r["ci95_atas"]:.4f}; sd = {r["std_val"]:.4f})')
    print(f'  ROC-AUC 25 fold : {a["mean_val"]:.4f} +/- {a["margin_error"]:.4f} '
          f'(CI 95%: {a["ci95_bawah"]:.4f} - {a["ci95_atas"]:.4f})')
    print(f'  Recall V2 (1x holdout) = {b["recall"]:.4f} -> masuk CI 95%? {di_dalam}')
    print(f'  Gap train-validasi (recall) = {r["gap_train_val"]:+.4f} '
          f'({"indikasi overfitting" if r["gap_train_val"] > 0.05 else "tidak ada indikasi overfitting"})')
    print()
print('Interpretasi: lebar interval kepercayaan menunjukkan bahwa selisih recall antar model')
print('sebesar 0.01-0.02 masih berada dalam rentang variasi pengambilan sampel, sehingga')
print('perbandingan model TIDAK boleh disimpulkan hanya dari satu angka holdout.')

In [ ]:
# ============================================================
# CELL 10: Grafik Eksperimen 1 - Boxplot 25 Skor Recall + Errorbar CI 95%
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(19, 5.6))

# (a) Boxplot distribusi 25 skor recall per model
data_box  = [hasil_repeated[n]['test_recall'] for n in PABRIK_MODEL]
label_box = list(PABRIK_MODEL.keys())
bp = axes[0].boxplot(data_box, patch_artist=True, widths=0.55,
                     medianprops=dict(color='black', linewidth=2))
axes[0].set_xticks(np.arange(1, len(label_box) + 1))
axes[0].set_xticklabels(label_box)
for patch, nama in zip(bp['boxes'], label_box):
    patch.set_facecolor(WARNA_MODEL[nama]); patch.set_alpha(0.65)
for i, nama in enumerate(label_box, start=1):
    v = hasil_repeated[nama]['test_recall']
    axes[0].scatter(np.random.normal(i, 0.045, len(v)), v, s=14, color='black', alpha=0.35, zorder=3)
    axes[0].scatter([i], [BASELINE_V2[nama]['recall']], marker='D', s=70,
                    color=WARNA_AKSEN, zorder=4,
                    label='Hasil V2 (1x holdout)' if i == 1 else None)
axes[0].set_title('(a) Distribusi 25 Skor Recall\nRepeatedStratifiedKFold 5x5')
axes[0].set_ylabel('Recall (validasi)')
axes[0].legend(loc='lower right', fontsize=9)
axes[0].tick_params(axis='x', rotation=8)

# (b) Errorbar rata-rata +/- CI 95% untuk seluruh metrik
posisi = np.arange(len(METRIK_CV))
for k, nama in enumerate(label_box):
    sub = tabel_repeated_cv[tabel_repeated_cv['model'] == nama].set_index('metrik').loc[METRIK_CV]
    off = (k - 1) * 0.22
    axes[1].errorbar(posisi + off, sub['mean_val'].values,
                     yerr=sub['margin_error'].values, fmt='o', capsize=5,
                     markersize=7, linewidth=2, color=WARNA_MODEL[nama], label=nama)
axes[1].set_xticks(posisi)
axes[1].set_xticklabels(['Recall', 'Precision', 'F1', 'ROC-AUC', 'PR-AUC'], rotation=12)
axes[1].set_title('(b) Rata-rata +/- Interval Kepercayaan 95%\n(distribusi t, df = 24)')
axes[1].set_ylabel('Nilai metrik')
axes[1].legend(fontsize=9)

# (c) Perbandingan skor latih vs validasi (deteksi overfitting)
lebar = 0.35
for k, nama in enumerate(label_box):
    sub = tabel_repeated_cv[(tabel_repeated_cv['model'] == nama)].set_index('metrik').loc[METRIK_CV]
    axes[2].bar(posisi + (k - 1) * 0.26, sub['gap_train_val'].values, width=0.24,
                color=WARNA_MODEL[nama], alpha=0.85, label=nama)
axes[2].axhline(0.05, color=WARNA_AKSEN, linestyle='--', linewidth=1.6,
                label='Ambang indikasi overfitting (0.05)')
axes[2].axhline(0, color='black', linewidth=1)
axes[2].set_xticks(posisi)
axes[2].set_xticklabels(['Recall', 'Precision', 'F1', 'ROC-AUC', 'PR-AUC'], rotation=12)
axes[2].set_title('(c) Selisih Skor Latih - Validasi\n(semakin kecil semakin baik)')
axes[2].set_ylabel('Gap train - validation')
axes[2].legend(fontsize=8)

plt.suptitle('EKSPERIMEN 1: Repeated Stratified K-Fold Cross Validation (5 fold x 5 repetisi)',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
simpan_gambar('stat_repeated_cv')
plt.show()

# Checkpoint hasil eksperimen 1
_ckpt_e1 = simpan_json(tabel_repeated_cv.to_dict('records'), 'checkpoint_repeated_cv')

---

# EKSPERIMEN 2 — Nested Cross Validation

**Masalah yang dijawab:** pada notebook V2, hyperparameter dipilih dengan
`GridSearchCV`/`RandomizedSearchCV` memakai 5-fold CV, kemudian skor CV terbaik
(`best_score_`) ikut dilaporkan sebagai performa model. Skor tersebut **bias optimistik**,
karena data yang dipakai untuk memilih hyperparameter adalah data yang sama yang dipakai
untuk menilai hasilnya.

**Desain nested CV:**

- **Outer loop** (`StratifiedKFold`, 5 fold) — hanya untuk *menilai*. Data uji outer tidak
  pernah tersentuh proses tuning.
- **Inner loop** (`StratifiedKFold`, 3 fold, di dalam `GridSearchCV`) — hanya untuk
  *memilih* hyperparameter, memakai data latih outer saja.

Selisih antara skor CV biasa (non-nested / *flat*) dan skor nested CV adalah estimasi
besarnya **bias optimistik** akibat tuning. Grid sengaja dibuat ringkas agar biaya
komputasi (5 x 3 x jumlah kombinasi fit per model) tetap wajar di Colab.


In [ ]:
# ============================================================
# CELL 11: EKSPERIMEN 2 - Nested Cross Validation (outer 5 fold, inner 3 fold)
# ============================================================
# Grid ringkas per model. Perhatikan penamaan parameter di dalam pipeline:
#   RF  : clf__<param>              (RandomForestClassifier bernama 'clf')
#   KNN : clf__<param>              (KNeighborsClassifier bernama 'clf')
#   SVM : clf__estimator__<param>   ('clf' = CalibratedClassifierCV, estimator = LinearSVC)
GRID_NESTED = {
    'Random Forest': {'clf__n_estimators': [100, 200], 'clf__max_depth': [8, 10, 14]},
    'KNN'          : {'clf__n_neighbors' : [11, 21, 31]},
    'SVM (Linear)' : {'clf__estimator__C': [0.01, 0.1, 1.0]},
}

def jalankan_nested_cv(nama, pabrik, grid, X, y, n_outer=5, n_inner=3, scoring='recall'):
    """
    Nested cross validation.
    Outer  : menilai performa (data uji outer tidak dipakai untuk tuning).
    Inner  : GridSearchCV memilih hyperparameter hanya dari data latih outer.
    Return : dict berisi skor tiap outer fold, parameter terpilih, dan ringkasannya.
    """
    outer = StratifiedKFold(n_splits=n_outer, shuffle=True, random_state=RANDOM_STATE)
    inner = StratifiedKFold(n_splits=n_inner, shuffle=True, random_state=RANDOM_STATE)

    skor_recall, skor_auc, skor_f1, param_terpilih, waktu_fold = [], [], [], [], []
    for i, (idx_tr, idx_te) in enumerate(outer.split(X, y), start=1):
        t0 = time.time()
        gs = GridSearchCV(pabrik(), grid, scoring=scoring, cv=inner, n_jobs=-1, refit=True)
        gs.fit(X.iloc[idx_tr], y.iloc[idx_tr])

        y_true_o  = y.iloc[idx_te]
        y_pred_o  = gs.predict(X.iloc[idx_te])
        y_proba_o = gs.predict_proba(X.iloc[idx_te])[:, 1]

        r = recall_score(y_true_o, y_pred_o, zero_division=0)
        f = f1_score(y_true_o, y_pred_o, zero_division=0)
        a = roc_auc_score(y_true_o, y_proba_o)
        dt = time.time() - t0

        skor_recall.append(r); skor_f1.append(f); skor_auc.append(a)
        param_terpilih.append(gs.best_params_); waktu_fold.append(dt)
        print(f'    fold {i}/{n_outer} | recall = {r:.4f} | auc = {a:.4f} | '
              f'param = {gs.best_params_} | {dt:.1f} s')

    return {'recall': skor_recall, 'f1': skor_f1, 'roc_auc': skor_auc,
            'params': param_terpilih, 'waktu': waktu_fold}

garis('EKSPERIMEN 2: NESTED CROSS VALIDATION')
print(f'Jumlah data  : {len(X_nest):,} baris')
print('Skema        : outer StratifiedKFold(5) x inner StratifiedKFold(3)')
print('Estimasi     : sekitar 8 - 15 menit (eksperimen paling lama di notebook ini).')
print('Progres dicetak per outer fold supaya terlihat notebook tidak menggantung.')
print()

hasil_nested, hasil_flat = {}, {}
t_mulai_e2 = time.time()

for nama, pabrik in PABRIK_MODEL.items():
    print(f'[NESTED] {nama} - grid: {GRID_NESTED[nama]}')
    hasil_nested[nama] = jalankan_nested_cv(nama, pabrik, GRID_NESTED[nama], X_nest, y_nest)

    # Skor CV non-nested (flat): tuning dan pelaporan memakai data yang sama -> bias optimistik
    t0 = time.time()
    gs_flat = GridSearchCV(pabrik(), GRID_NESTED[nama], scoring='recall',
                           cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
                           n_jobs=-1, refit=False)
    gs_flat.fit(X_nest, y_nest)
    hasil_flat[nama] = {'best_score': float(gs_flat.best_score_),
                        'best_params': gs_flat.best_params_}
    print(f'    [FLAT / non-nested] recall = {gs_flat.best_score_:.4f} | '
          f'param = {gs_flat.best_params_} | {time.time()-t0:.1f} s')
    print(f'    Bias optimistik = {gs_flat.best_score_ - np.mean(hasil_nested[nama]["recall"]):+.4f}')
    print()

print(f'Total waktu Eksperimen 2: {(time.time() - t_mulai_e2)/60:.2f} menit')

In [ ]:
# ============================================================
# CELL 12: Tabel & Grafik Nested CV vs CV Non-Nested (bias optimistik tuning)
# ============================================================
baris_nested = []
for nama in PABRIK_MODEL:
    h  = hasil_nested[nama]
    c  = ci95_t(h['recall'])
    ca = ci95_t(h['roc_auc'])
    flat = hasil_flat[nama]['best_score']
    baris = {
        'model'              : nama,
        'nested_recall_mean' : round(c['mean'], 4),
        'nested_recall_std'  : round(c['std'], 4),
        'nested_ci95_bawah'  : round(c['ci_bawah'], 4),
        'nested_ci95_atas'   : round(c['ci_atas'], 4),
        'nested_auc_mean'    : round(ca['mean'], 4),
        'flat_cv_recall'     : round(flat, 4),
        'bias_optimistik'    : round(flat - c['mean'], 4),
        'recall_v2_holdout'  : BASELINE_V2[nama]['recall'],
        'param_paling_sering': str(max(map(str, h['params']), key=list(map(str, h['params'])).count)),
        'total_waktu_s'      : round(float(np.sum(h['waktu'])), 1),
        'pelaporan'          : f"{c['mean']:.4f} +/- {c['margin']:.4f}",
    }
    for i, v in enumerate(h['recall'], start=1):
        baris[f'fold_{i}'] = round(float(v), 4)
    baris_nested.append(baris)

tabel_nested_cv = pd.DataFrame(baris_nested)
simpan_tabel(tabel_nested_cv, 'tabel_nested_cv')

fig, axes = plt.subplots(1, 2, figsize=(16, 5.6))

# (a) Nested vs flat vs holdout V2
nm = list(PABRIK_MODEL.keys())
x  = np.arange(len(nm)); w = 0.26
nested_mean = tabel_nested_cv['nested_recall_mean'].values
nested_err  = [(tabel_nested_cv['nested_recall_mean'] - tabel_nested_cv['nested_ci95_bawah']).values,
               (tabel_nested_cv['nested_ci95_atas'] - tabel_nested_cv['nested_recall_mean']).values]
axes[0].bar(x - w, nested_mean, w, yerr=nested_err, capsize=5,
            color=[WARNA_MODEL[n] for n in nm], label='Nested CV (tak bias)')
axes[0].bar(x, tabel_nested_cv['flat_cv_recall'].values, w,
            color=WARNA_AKSEN, alpha=0.9, label='CV non-nested (bias optimistik)')
axes[0].bar(x + w, tabel_nested_cv['recall_v2_holdout'].values, w,
            color='#7f8c8d', alpha=0.85, label='Holdout 1x (notebook V2)')
for i in range(len(nm)):
    axes[0].text(i, tabel_nested_cv['flat_cv_recall'].values[i] + 0.006,
                 f'+{tabel_nested_cv["bias_optimistik"].values[i]:.4f}',
                 ha='center', fontsize=9, color='#b9770e', fontweight='bold')
axes[0].set_xticks(x); axes[0].set_xticklabels(nm, rotation=8)
axes[0].set_ylabel('Recall')
axes[0].set_ylim(min(0.80, nested_mean.min() - 0.05), 1.02)
axes[0].set_title('(a) Nested CV vs CV Non-Nested vs Holdout V2\n(angka oranye = besar bias optimistik)')
axes[0].legend(fontsize=9, loc='lower right')

# (b) Skor recall tiap outer fold
for nama in nm:
    axes[1].plot(range(1, 6), hasil_nested[nama]['recall'], marker='o', linewidth=2,
                 color=WARNA_MODEL[nama], label=nama)
    axes[1].axhline(np.mean(hasil_nested[nama]['recall']), color=WARNA_MODEL[nama],
                    linestyle=':', alpha=0.6)
axes[1].set_xticks(range(1, 6))
axes[1].set_xlabel('Outer fold ke-'); axes[1].set_ylabel('Recall outer')
axes[1].set_title('(b) Recall per Outer Fold\n(garis putus-putus = rata-rata nested)')
axes[1].legend(fontsize=9)

plt.suptitle('EKSPERIMEN 2: Nested Cross Validation (outer 5 x inner 3)',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
simpan_gambar('stat_nested_vs_flat')
plt.show()

garis('KESIMPULAN EKSPERIMEN 2 (siap salin ke skripsi)')
for r in tabel_nested_cv.to_dict('records'):
    print(f'{r["model"]}')
    print(f'  Nested CV recall      : {r["nested_recall_mean"]:.4f} '
          f'(CI 95%: {r["nested_ci95_bawah"]:.4f} - {r["nested_ci95_atas"]:.4f}; '
          f'sd = {r["nested_recall_std"]:.4f})')
    print(f'  CV non-nested (flat)  : {r["flat_cv_recall"]:.4f}')
    print(f'  Bias optimistik tuning: {r["bias_optimistik"]:+.4f}')
    print(f'  Hyperparameter paling sering terpilih: {r["param_paling_sering"]}')
    print()
rata_bias = tabel_nested_cv['bias_optimistik'].mean()
print(f'Rata-rata bias optimistik ketiga model: {rata_bias:+.4f}')
print('Interpretasi: skor cross validation yang dilaporkan langsung dari proses tuning')
print('cenderung lebih tinggi daripada estimasi nested CV. Karena besarnya bias tergolong')
print('kecil, kesimpulan pemilihan model pada notebook V2 tetap valid, namun angka yang')
print('lebih jujur untuk dilaporkan dalam skripsi adalah hasil nested CV di atas.')

_ckpt_e2 = simpan_json(tabel_nested_cv.to_dict('records'), 'checkpoint_nested_cv')

---

# EKSPERIMEN 3 — Uji Statistik Perbandingan Antar Model

Notebook V2 hanya memakai satu uji (McNemar). Notebook ini memakai **empat uji** yang
saling melengkapi karena masing-masing mengukur aspek berbeda:

| Uji | Data yang diuji | Yang diukur | Referensi |
|---|---|---|---|
| **McNemar** | prediksi biner pada test set | apakah pola kesalahan dua model berbeda | McNemar (1947); Dietterich (1998) |
| **5x2cv paired t-test** | 10 skor dari 5 pengulangan 2-fold | perbedaan performa dengan memperhitungkan variasi data latih | Dietterich (1998) |
| **Wilcoxon signed-rank** | 25 skor per-fold repeated CV | perbedaan performa tanpa asumsi normalitas | Demsar (2006) |
| **DeLong** | skor probabilitas kontinu | perbedaan ROC-AUC (memakai teori U-statistic) | DeLong et al. (1988); Sun & Xu (2014) |

Hipotesis nol untuk semua uji: **tidak ada perbedaan performa antara kedua model**.
Taraf signifikansi yang dipakai: alpha = 0.05.

Perbandingan dilakukan pada operating point yang benar-benar dipakai sistem, yaitu
prediksi hasil *thresholding* Youden per model, bukan threshold default 0.5.


In [ ]:
# ============================================================
# CELL 13: Melatih 3 Model pada Data Latih & Prediksi Test Set
# (dasar untuk McNemar, DeLong, analisis threshold, dan kalibrasi)
# ============================================================
garis('MELATIH MODEL FINAL PADA SPLIT 80:20 UNTUK UJI STATISTIK')
print(f'Train: {len(X_train):,} baris | Test: {len(X_test):,} baris '
      f'({int(y_test.sum()):,} positif)')
print()

model_terlatih, proba_test, pred_test, thr_youden = {}, {}, {}, {}

for nama, pabrik in PABRIK_MODEL.items():
    t0 = time.time()
    m = pabrik()
    m.fit(X_train, y_train)
    p = m.predict_proba(X_test)[:, 1]
    thr = threshold_youden(y_test_np, p)

    model_terlatih[nama] = m
    proba_test[nama]     = p
    thr_youden[nama]     = thr
    pred_test[nama]      = (p >= thr).astype(int)

    met = hitung_metrik(y_test_np, pred_test[nama], p)
    print(f'{nama:14s} | thr Youden = {thr:.4f} | recall = {met["recall"]:.4f} | '
          f'precision = {met["precision"]:.4f} | AUC = {met["roc_auc"]:.4f} | '
          f'{time.time()-t0:.1f} s')

print()
print('Threshold Youden hasil notebook V2 sebagai pembanding:')
for nama in PABRIK_MODEL:
    print(f'  {nama:14s} : V3 = {thr_youden[nama]:.4f} | V2 = {BASELINE_V2[nama]["threshold"]:.4f}')

PASANGAN = [('Random Forest', 'KNN'), ('Random Forest', 'SVM (Linear)'), ('KNN', 'SVM (Linear)')]
ALPHA = 0.05
hasil_uji = []   # dikumpulkan dari CELL 14-18

In [ ]:
# ============================================================
# CELL 14: UJI 1 - McNemar's Test + Odds Ratio + CI 95%
# ============================================================
def uji_mcnemar(y_true, pred_a, pred_b, nama_a, nama_b, alpha=0.05):
    """
    McNemar's test untuk dua classifier pada test set yang sama.
    Tabel kontingensi disusun dari status benar/salah tiap model:
        n11 = kedua model benar
        n10 = A benar, B salah   (disebut b)
        n01 = A salah, B benar   (disebut c)
        n00 = kedua model salah
    H0: b = c (kedua model punya proporsi kesalahan yang sama).
    Statistik memakai koreksi kontinuitas bila b + c > 25, selain itu memakai uji binomial eksak.
    Odds ratio = b / c dengan CI 95% dari se(log OR) = sqrt(1/b + 1/c).
    """
    benar_a = (np.asarray(pred_a) == np.asarray(y_true))
    benar_b = (np.asarray(pred_b) == np.asarray(y_true))
    n11 = int(np.sum(benar_a & benar_b))
    n10 = int(np.sum(benar_a & ~benar_b))     # b
    n01 = int(np.sum(~benar_a & benar_b))     # c
    n00 = int(np.sum(~benar_a & ~benar_b))
    tabel = np.array([[n11, n10], [n01, n00]])

    eksak = (n10 + n01) < 25
    res   = mcnemar(tabel, exact=eksak, correction=not eksak)
    stat, p = float(res.statistic), float(res.pvalue)

    if n10 > 0 and n01 > 0:
        odds  = n10 / n01
        se_lg = math.sqrt(1.0 / n10 + 1.0 / n01)
        or_lo = math.exp(math.log(odds) - 1.96 * se_lg)
        or_hi = math.exp(math.log(odds) + 1.96 * se_lg)
    else:
        odds, or_lo, or_hi = np.nan, np.nan, np.nan

    signifikan = p < alpha
    lebih_baik = nama_a if n10 > n01 else nama_b
    return {
        'pasangan'  : f'{nama_a} vs {nama_b}',
        'uji'       : 'McNemar',
        'statistik' : round(stat, 4),
        'p_value'   : p,
        'detail'    : (f'n11={n11}, b(A benar/B salah)={n10}, c(A salah/B benar)={n01}, '
                       f'n00={n00}, OR={odds:.3f} [CI95%: {or_lo:.3f}-{or_hi:.3f}], '
                       f'{"eksak" if eksak else "chi-square + koreksi kontinuitas"}'),
        'kesimpulan': (f'Berbeda signifikan (p < {alpha}); {lebih_baik} lebih sedikit salah'
                       if signifikan else f'Tidak berbeda signifikan (p >= {alpha})'),
        'signifikan': bool(signifikan),
        'n11': n11, 'b': n10, 'c': n01, 'n00': n00,
        'odds_ratio': odds, 'or_ci_bawah': or_lo, 'or_ci_atas': or_hi,
    }

garis('UJI 1: McNEMAR TEST (prediksi test set pada threshold Youden)')
hasil_mcnemar = []
for a, b in PASANGAN:
    r = uji_mcnemar(y_test_np, pred_test[a], pred_test[b], a, b, ALPHA)
    hasil_mcnemar.append(r)
    hasil_uji.append({k: r[k] for k in ['pasangan', 'uji', 'statistik', 'p_value',
                                        'detail', 'kesimpulan', 'signifikan']})
    print(f'{r["pasangan"]}')
    print(f'  statistik = {r["statistik"]:.4f} | p-value = {r["p_value"]:.6f}')
    print(f'  {r["detail"]}')
    print(f'  -> {r["kesimpulan"]}')
    print()

In [ ]:
# ============================================================
# CELL 15: UJI 2 - 5x2cv Paired t-test (Dietterich, 1998) - implementasi manual
# ============================================================
def _skor_metrik(metrik, y_true, y_pred):
    if metrik == 'recall':
        return recall_score(y_true, y_pred, zero_division=0)
    if metrik == 'f1':
        return f1_score(y_true, y_pred, zero_division=0)
    return accuracy_score(y_true, y_pred)

def uji_5x2cv_paired_t(pabrik_a, pabrik_b, X, y, nama_a, nama_b,
                       metrik='recall', alpha=0.05, random_state=RANDOM_STATE, verbose=True):
    """
    5x2cv paired t-test (Dietterich, 1998, Neural Computation 10(7):1895-1923).

    Prosedur:
      Ulangi 5 kali (i = 1..5):
        - Bagi data menjadi dua bagian 50:50 secara stratified (fold A dan fold B).
        - Putaran 1: latih di A, uji di B  -> selisih skor p_i^(1) = skor_A_model - skor_B_model
        - Putaran 2: latih di B, uji di A  -> selisih skor p_i^(2)
        - p_bar_i = (p_i^(1) + p_i^(2)) / 2
        - s_i^2   = (p_i^(1) - p_bar_i)^2 + (p_i^(2) - p_bar_i)^2

      Statistik uji:
                        p_1^(1)
        t = ------------------------------- ,   df = 5
             sqrt( (1/5) * sum_{i=1..5} s_i^2 )

    Uji ini dirancang khusus untuk perbandingan algoritma karena memperhitungkan
    variasi akibat perbedaan data latih, sesuatu yang tidak ditangkap McNemar
    (McNemar hanya melihat satu model terlatih pada satu test set).
    """
    Xv = X.reset_index(drop=True); yv = y.reset_index(drop=True)
    p_pertama, variansi, semua_selisih = None, [], []

    for i in range(5):
        X_a, X_b, y_a, y_b = train_test_split(
            Xv, yv, test_size=0.5, stratify=yv, random_state=random_state + i)

        # Putaran 1: latih pada fold A, uji pada fold B
        ma = pabrik_a(); ma.fit(X_a, y_a)
        mb = pabrik_b(); mb.fit(X_a, y_a)
        p1 = (_skor_metrik(metrik, y_b, ma.predict(X_b)) -
              _skor_metrik(metrik, y_b, mb.predict(X_b)))

        # Putaran 2: latih pada fold B, uji pada fold A
        ma2 = pabrik_a(); ma2.fit(X_b, y_b)
        mb2 = pabrik_b(); mb2.fit(X_b, y_b)
        p2 = (_skor_metrik(metrik, y_a, ma2.predict(X_a)) -
              _skor_metrik(metrik, y_a, mb2.predict(X_a)))

        p_bar = (p1 + p2) / 2.0
        s2    = (p1 - p_bar) ** 2 + (p2 - p_bar) ** 2
        variansi.append(s2); semua_selisih.extend([p1, p2])
        if i == 0:
            p_pertama = p1
        if verbose:
            print(f'    replikasi {i+1}/5 | p(1) = {p1:+.4f} | p(2) = {p2:+.4f} | s^2 = {s2:.6f}')

    penyebut = math.sqrt(max(np.mean(variansi), 1e-12))
    t_stat   = float(p_pertama / penyebut)
    p_value  = float(2.0 * sstats.t.sf(abs(t_stat), df=5))
    signifikan = p_value < alpha
    lebih_baik = nama_a if np.mean(semua_selisih) > 0 else nama_b
    return {
        'pasangan'  : f'{nama_a} vs {nama_b}',
        'uji'       : '5x2cv paired t-test',
        'statistik' : round(t_stat, 4),
        'p_value'   : p_value,
        'detail'    : (f'metrik = {metrik}, df = 5, selisih rata-rata 10 fold = '
                       f'{np.mean(semua_selisih):+.4f}, sd selisih = {np.std(semua_selisih, ddof=1):.4f}'),
        'kesimpulan': (f'Berbeda signifikan (p < {alpha}); {lebih_baik} lebih unggul'
                       if signifikan else f'Tidak berbeda signifikan (p >= {alpha})'),
        'signifikan': bool(signifikan),
        'selisih_rata2': float(np.mean(semua_selisih)),
    }

garis('UJI 2: 5x2cv PAIRED t-TEST (Dietterich, 1998)')
print(f'Data: {len(X_52):,} baris | 5 replikasi x 2 fold x 2 model x 3 pasangan')
print('Estimasi waktu: sekitar 2 - 5 menit.')
print()

hasil_52cv = []
t_mulai_52 = time.time()
for a, b in PASANGAN:
    print(f'[5x2cv] {a} vs {b}')
    r = uji_5x2cv_paired_t(PABRIK_MODEL[a], PABRIK_MODEL[b], X_52, y_52, a, b,
                           metrik='recall', alpha=ALPHA)
    hasil_52cv.append(r)
    hasil_uji.append({k: r[k] for k in ['pasangan', 'uji', 'statistik', 'p_value',
                                        'detail', 'kesimpulan', 'signifikan']})
    print(f'  t = {r["statistik"]:.4f} | p-value = {r["p_value"]:.6f} -> {r["kesimpulan"]}')
    print()
print(f'Total waktu uji 5x2cv: {(time.time() - t_mulai_52)/60:.2f} menit')

In [ ]:
# ============================================================
# CELL 16: UJI 3 - Wilcoxon Signed-Rank pada 25 Skor Repeated CV
# ============================================================
def uji_wilcoxon(skor_a, skor_b, nama_a, nama_b, metrik, alpha=0.05):
    """
    Wilcoxon signed-rank test (uji non-parametrik berpasangan).
    Dipakai pada 25 skor per-fold dari Eksperimen 1. Karena RepeatedStratifiedKFold
    memakai random_state tetap, pembagian fold untuk ketiga model identik sehingga
    skor benar-benar berpasangan.
    Uji ini tidak mengasumsikan sebaran selisih berdistribusi normal (Demsar, 2006).
    """
    a = np.asarray(skor_a, dtype=float); b = np.asarray(skor_b, dtype=float)
    selisih = a - b
    if np.allclose(selisih, 0):
        return {'pasangan': f'{nama_a} vs {nama_b}', 'uji': f'Wilcoxon ({metrik})',
                'statistik': 0.0, 'p_value': 1.0,
                'detail': 'seluruh selisih bernilai nol', 'kesimpulan': 'Identik',
                'signifikan': False}
    stat, p = wilcoxon(a, b, zero_method='wilcox', alternative='two-sided')
    n_menang = int(np.sum(selisih > 0)); n_kalah = int(np.sum(selisih < 0))
    signifikan = p < alpha
    lebih_baik = nama_a if np.median(selisih) > 0 else nama_b
    return {
        'pasangan'  : f'{nama_a} vs {nama_b}',
        'uji'       : f'Wilcoxon ({metrik})',
        'statistik' : round(float(stat), 4),
        'p_value'   : float(p),
        'detail'    : (f'n = {len(a)} fold berpasangan, median selisih = {np.median(selisih):+.4f}, '
                       f'menang/kalah = {n_menang}/{n_kalah}'),
        'kesimpulan': (f'Berbeda signifikan (p < {alpha}); {lebih_baik} lebih unggul'
                       if signifikan else f'Tidak berbeda signifikan (p >= {alpha})'),
        'signifikan': bool(signifikan),
    }

garis('UJI 3: WILCOXON SIGNED-RANK (25 skor per-fold Repeated CV)')
hasil_wilcoxon = []
for metrik in ['recall', 'roc_auc']:
    print(f'-- metrik: {metrik} --')
    for a, b in PASANGAN:
        r = uji_wilcoxon(hasil_repeated[a][f'test_{metrik}'],
                         hasil_repeated[b][f'test_{metrik}'], a, b, metrik, ALPHA)
        hasil_wilcoxon.append(r)
        hasil_uji.append(r)
        print(f'  {r["pasangan"]:32s} | W = {r["statistik"]:>9.2f} | '
              f'p = {r["p_value"]:.6f} -> {r["kesimpulan"]}')
    print()

In [ ]:
# ============================================================
# CELL 17: UJI 4 - DeLong Test untuk Perbedaan ROC-AUC (implementasi manual)
# ============================================================
def _midrank(x):
    """
    Midrank: peringkat 1..n dengan nilai kembar (ties) diberi rata-rata peringkatnya.
    Diperlukan agar estimasi komponen U-statistic tetap benar ketika banyak
    probabilitas prediksi bernilai identik (sering terjadi pada Random Forest).
    """
    J = np.argsort(x, kind='mergesort')
    Z = np.asarray(x, dtype=float)[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1)
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T + 1
    return T2

def _fast_delong(preds_sorted, n_positif):
    """
    Algoritma fast DeLong (Sun & Xu, 2014, IEEE Signal Processing Letters 21(11):1389-1393),
    versi cepat O(n log n) dari estimator kovarians DeLong et al. (1988, Biometrics 44:837-845).

    Parameter
    ---------
    preds_sorted : ndarray (k, n)
        k baris skor prediksi (k model), sudah diurutkan sehingga n_positif kolom
        pertama adalah sampel kelas positif dan sisanya kelas negatif.
    n_positif : int
        Banyaknya sampel positif (m).

    Return
    ------
    aucs : ndarray (k,)      nilai AUC tiap model
    cov  : ndarray (k, k)    matriks kovarians antar AUC

    Dasar teori: AUC identik dengan Mann-Whitney U statistic. Komponen struktural
    (placement values) V10 dan V01 dihitung dari midrank, lalu kovarians AUC diperoleh
    dari S10/m + S01/n.
    """
    m = int(n_positif)
    n = preds_sorted.shape[1] - m
    positif = preds_sorted[:, :m]
    negatif = preds_sorted[:, m:]
    k = preds_sorted.shape[0]

    tx = np.empty([k, m], dtype=float)
    ty = np.empty([k, n], dtype=float)
    tz = np.empty([k, m + n], dtype=float)
    for r in range(k):
        tx[r, :] = _midrank(positif[r, :])
        ty[r, :] = _midrank(negatif[r, :])
        tz[r, :] = _midrank(preds_sorted[r, :])

    aucs = tz[:, :m].sum(axis=1) / m / n - float(m + 1.0) / 2.0 / n
    v01  = (tz[:, :m] - tx[:, :]) / n            # placement value sampel positif
    v10  = 1.0 - (tz[:, m:] - ty[:, :]) / m      # placement value sampel negatif
    s01  = np.cov(v01)
    s10  = np.cov(v10)
    cov  = s01 / m + s10 / n
    return aucs, np.atleast_2d(cov)

def uji_delong(y_true, proba_a, proba_b, nama_a, nama_b, alpha=0.05):
    """
    DeLong test: uji signifikansi selisih dua ROC-AUC yang dihitung pada test set
    yang sama (berkorelasi). Statistik uji:

        z = (AUC_A - AUC_B) / sqrt( var(AUC_A) + var(AUC_B) - 2*cov(AUC_A, AUC_B) )

    dengan z ~ N(0, 1) di bawah H0: AUC_A = AUC_B. Suku kovarians itulah yang membuat
    uji ini tepat untuk dua model yang dievaluasi pada data uji yang sama; memakai uji
    dua sampel biasa akan melebih-lebihkan ragam dan menurunkan daya uji.

    Return: dict berisi AUC kedua model, selisih, standard error, z-score, p-value,
    dan CI 95% untuk selisih AUC.
    """
    y = np.asarray(y_true).astype(int)
    assert set(np.unique(y)).issubset({0, 1}), 'y_true harus biner 0/1'
    urutan   = (-y).argsort(kind='mergesort')      # kelas positif diletakkan di depan
    m        = int(y.sum())
    preds    = np.vstack((np.asarray(proba_a, dtype=float),
                          np.asarray(proba_b, dtype=float)))[:, urutan]
    aucs, cov = _fast_delong(preds, m)

    l    = np.array([[1.0, -1.0]])
    var  = float(np.asarray(l.dot(cov).dot(l.T)).reshape(-1)[0])
    se   = math.sqrt(max(var, 1e-30))
    beda = float(aucs[0] - aucs[1])
    z    = beda / se
    p    = float(2.0 * norm.sf(abs(z)))
    signifikan = p < alpha
    lebih_baik = nama_a if beda > 0 else nama_b
    return {
        'pasangan'  : f'{nama_a} vs {nama_b}',
        'uji'       : 'DeLong (ROC-AUC)',
        'statistik' : round(z, 4),
        'p_value'   : p,
        'detail'    : (f'AUC {nama_a} = {aucs[0]:.4f}, AUC {nama_b} = {aucs[1]:.4f}, '
                       f'selisih = {beda:+.4f} (SE = {se:.5f}; '
                       f'CI 95%: {beda - 1.96*se:+.4f} sampai {beda + 1.96*se:+.4f})'),
        'kesimpulan': (f'AUC berbeda signifikan (p < {alpha}); {lebih_baik} lebih unggul'
                       if signifikan else f'AUC tidak berbeda signifikan (p >= {alpha})'),
        'signifikan': bool(signifikan),
        'auc_a': float(aucs[0]), 'auc_b': float(aucs[1]),
        'selisih_auc': beda, 'se': se,
        'ci_bawah': beda - 1.96 * se, 'ci_atas': beda + 1.96 * se,
    }

garis('UJI 4: DeLONG TEST (perbedaan ROC-AUC pada test set yang sama)')
# Validasi implementasi: AUC hasil fungsi DeLong harus sama dengan roc_auc_score sklearn
_cek = uji_delong(y_test_np, proba_test['Random Forest'], proba_test['KNN'],
                  'Random Forest', 'KNN', ALPHA)
print('Validasi implementasi (AUC DeLong vs sklearn roc_auc_score):')
print(f'  Random Forest : DeLong = {_cek["auc_a"]:.6f} | '
      f'sklearn = {roc_auc_score(y_test_np, proba_test["Random Forest"]):.6f}')
print(f'  KNN           : DeLong = {_cek["auc_b"]:.6f} | '
      f'sklearn = {roc_auc_score(y_test_np, proba_test["KNN"]):.6f}')
print()

hasil_delong = []
for a, b in PASANGAN:
    r = uji_delong(y_test_np, proba_test[a], proba_test[b], a, b, ALPHA)
    hasil_delong.append(r)
    hasil_uji.append({k: r[k] for k in ['pasangan', 'uji', 'statistik', 'p_value',
                                        'detail', 'kesimpulan', 'signifikan']})
    print(f'{r["pasangan"]}')
    print(f'  z = {r["statistik"]:.4f} | p-value = {r["p_value"]:.6e}')
    print(f'  {r["detail"]}')
    print(f'  -> {r["kesimpulan"]}')
    print()

In [ ]:
# ============================================================
# CELL 18: Gabungan Seluruh Uji Statistik + Heatmap p-value
# ============================================================
tabel_uji_statistik = pd.DataFrame(hasil_uji)[
    ['pasangan', 'uji', 'statistik', 'p_value', 'detail', 'kesimpulan', 'signifikan']]
tabel_uji_statistik['p_value'] = tabel_uji_statistik['p_value'].astype(float).round(8)
simpan_tabel(tabel_uji_statistik, 'tabel_uji_statistik')

# Matriks p-value: baris = pasangan model, kolom = jenis uji
pivot_p = tabel_uji_statistik.pivot_table(index='pasangan', columns='uji',
                                          values='p_value', aggfunc='first')
urutan_uji = [u for u in ['McNemar', '5x2cv paired t-test', 'Wilcoxon (recall)',
                          'Wilcoxon (roc_auc)', 'DeLong (ROC-AUC)'] if u in pivot_p.columns]
pivot_p = pivot_p[urutan_uji]

fig, axes = plt.subplots(1, 2, figsize=(17, 5.2),
                         gridspec_kw={'width_ratios': [1.35, 1]})

sns.heatmap(pivot_p, annot=True, fmt='.4f', cmap='RdYlGn', vmin=0.0, vmax=0.10,
            linewidths=1.2, linecolor='white', ax=axes[0],
            cbar_kws={'label': 'p-value (dipotong pada 0.10)'})
axes[0].set_title('(a) Matriks p-value Seluruh Uji Statistik\n'
                  'merah = p < 0.05 (berbeda signifikan), hijau = tidak signifikan')
axes[0].set_xlabel('Jenis uji'); axes[0].set_ylabel('Pasangan model')
axes[0].tick_params(axis='x', rotation=20)

# Perbandingan ROC ketiga model + hasil DeLong
for nama in PABRIK_MODEL:
    fpr, tpr, _ = roc_curve(y_test_np, proba_test[nama])
    axes[1].plot(fpr, tpr, linewidth=2.2, color=WARNA_MODEL[nama],
                 label=f'{nama} (AUC = {roc_auc_score(y_test_np, proba_test[nama]):.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.6, label='Tebakan acak')
teks_delong = '\n'.join([f'{r["pasangan"]}: p = {r["p_value"]:.2e}' for r in hasil_delong])
axes[1].text(0.42, 0.18, 'Uji DeLong:\n' + teks_delong, fontsize=8.5,
             bbox=dict(boxstyle='round', facecolor='#fdf2e0', edgecolor=WARNA_AKSEN))
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('(b) Kurva ROC & Hasil Uji DeLong')
axes[1].legend(loc='lower right', fontsize=9)

plt.suptitle('EKSPERIMEN 3: Uji Statistik Perbandingan Antar Model (alpha = 0.05)',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
simpan_gambar('stat_matriks_uji')
plt.show()

garis('KESIMPULAN EKSPERIMEN 3 (siap salin ke skripsi)')
n_sig = int(tabel_uji_statistik['signifikan'].sum())
print(f'Total uji dilakukan   : {len(tabel_uji_statistik)} '
      f'({len(PASANGAN)} pasangan model x {tabel_uji_statistik["uji"].nunique()} jenis uji)')
print(f'Uji signifikan (<0.05): {n_sig}')
print()
for pas in tabel_uji_statistik['pasangan'].unique():
    sub = tabel_uji_statistik[tabel_uji_statistik['pasangan'] == pas]
    n_s = int(sub['signifikan'].sum())
    print(f'{pas}')
    for r in sub.to_dict('records'):
        tanda = 'SIGNIFIKAN    ' if r['signifikan'] else 'tidak signifikan'
        print(f'  {r["uji"]:22s} | stat = {r["statistik"]:>10.4f} | '
              f'p = {r["p_value"]:.6f} | {tanda}')
    print(f'  Ringkas: {n_s} dari {len(sub)} uji menyatakan berbeda signifikan.')
    print()
print('Catatan penting untuk skripsi: signifikansi statistik pada dataset besar sangat mudah')
print('tercapai walaupun selisih praktisnya kecil. Karena itu setiap kesimpulan uji di atas')
print('harus dibaca bersama ukuran efeknya (selisih recall/AUC beserta interval kepercayaan),')
print('bukan hanya dari nilai p-value.')

_ckpt_e3 = simpan_json(tabel_uji_statistik.to_dict('records'), 'checkpoint_uji_statistik')

---

# EKSPERIMEN 4 — Perbandingan Strategi Penentuan Threshold

**Masalah yang dijawab:** website DiaPredict memakai threshold **0.4965**. Pada notebook V2
nilai ini muncul begitu saja sebagai keluaran Youden's J tanpa pembanding, sehingga terkesan
angka ajaib. Eksperimen ini membandingkan **delapan strategi** penentuan threshold yang lazim
dipakai di literatur, lalu menunjukkan konsekuensi masing-masing terhadap jumlah pasien yang
lolos deteksi (False Negative) dan jumlah pasien sehat yang dirujuk sia-sia (False Positive).

| Kode | Strategi | Rumus / kriteria |
|---|---|---|
| (a) | Default | 0.5 |
| (b) | **Youden's J** | maksimum (sensitivitas + spesifisitas - 1) = maksimum (TPR - FPR) |
| (c) | F1 maksimum | maksimum harmonic mean precision-recall |
| (d) | Precision >= 0.50 | recall tertinggi dengan syarat precision minimal 0.50 |
| (e) | Recall >= 0.90 | precision tertinggi dengan syarat recall minimal 0.90 |
| (f) | Cost-sensitive 5:1 | minimum (5 x FN + 1 x FP) |
| (g) | Cost-sensitive 10:1 | minimum (10 x FN + 1 x FP) |
| (h) | Closest to (0,1) | minimum jarak Euclidean ke titik ROC ideal |

Rasio biaya 5:1 dan 10:1 mewakili asumsi bahwa **melewatkan pasien diabetes jauh lebih
merugikan** daripada merujuk orang sehat untuk pemeriksaan lanjutan: pasien yang tidak
terdeteksi berisiko mengalami komplikasi (retinopati, nefropati, neuropati), sementara
false positive hanya berkonsekuensi satu kali tes gula darah tambahan.


In [ ]:
# ============================================================
# CELL 19: EKSPERIMEN 4 - Fungsi Delapan Strategi Penentuan Threshold
# ============================================================
GRID_THR = np.round(np.arange(0.05, 0.9501, 0.005), 4)   # 181 kandidat threshold
BIAYA_FN_FP = [(5, 1), (10, 1)]

def metrik_pada_threshold(y_true, y_proba, thr):
    """Hitung metrik lengkap + jumlah FN/FP + total biaya pada satu nilai threshold."""
    y_pred = (np.asarray(y_proba) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    spes = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return {
        'threshold': float(thr),
        'accuracy' : (tp + tn) / len(y_true),
        'precision': prec, 'recall': rec, 'f1': f1, 'specificity': spes,
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn),
        'biaya_5_1' : int(5 * fn + 1 * fp),
        'biaya_10_1': int(10 * fn + 1 * fp),
    }

def cari_semua_threshold(y_true, y_proba):
    """
    Kembalikan dict {kode_strategi: (nama_strategi, nilai_threshold)} untuk 8 strategi.
    Strategi (b) dan (h) dihitung dari titik-titik kurva ROC yang sesungguhnya,
    strategi lain dicari lewat pemindaian grid threshold 0.05-0.95 (langkah 0.005).
    """
    y_true = np.asarray(y_true).astype(int)
    fpr, tpr, thr_roc = roc_curve(y_true, y_proba)
    thr_roc = np.clip(thr_roc, 0.0, 1.0)          # sklearn memberi inf pada elemen pertama

    baris = [metrik_pada_threshold(y_true, y_proba, t) for t in GRID_THR]
    dfg   = pd.DataFrame(baris)

    # (b) Youden's J = argmax(TPR - FPR)
    thr_b = float(thr_roc[np.argmax(tpr - fpr)])
    # (c) F1 maksimum
    thr_c = float(dfg.loc[dfg['f1'].idxmax(), 'threshold'])
    # (d) precision >= 0.50 dengan recall maksimum
    kand_d = dfg[dfg['precision'] >= 0.50]
    thr_d  = float(kand_d.loc[kand_d['recall'].idxmax(), 'threshold']) if len(kand_d) \
             else float(dfg.loc[dfg['precision'].idxmax(), 'threshold'])
    # (e) recall >= 0.90 dengan precision maksimum
    kand_e = dfg[dfg['recall'] >= 0.90]
    thr_e  = float(kand_e.loc[kand_e['precision'].idxmax(), 'threshold']) if len(kand_e) \
             else float(dfg.loc[dfg['recall'].idxmax(), 'threshold'])
    # (f) & (g) cost-sensitive
    thr_f = float(dfg.loc[dfg['biaya_5_1'].idxmin(), 'threshold'])
    thr_g = float(dfg.loc[dfg['biaya_10_1'].idxmin(), 'threshold'])
    # (h) titik ROC terdekat ke (0, 1)
    jarak = np.sqrt(fpr ** 2 + (1 - tpr) ** 2)
    thr_h = float(thr_roc[np.argmin(jarak)])

    return {
        'a': ('(a) Default 0.5', 0.5),
        'b': ("(b) Youden's J", thr_b),
        'c': ('(c) F1 maksimum', thr_c),
        'd': ('(d) Precision >= 0.50, recall maks', thr_d),
        'e': ('(e) Recall >= 0.90, precision maks', thr_e),
        'f': ('(f) Cost-sensitive FN:FP = 5:1', thr_f),
        'g': ('(g) Cost-sensitive FN:FP = 10:1', thr_g),
        'h': ('(h) Terdekat ke titik (0,1) ROC', thr_h),
    }, dfg

garis('EKSPERIMEN 4: PERBANDINGAN 8 STRATEGI PENENTUAN THRESHOLD')
print(f'Data uji: {len(y_test_np):,} baris | positif = {int(y_test_np.sum()):,} '
      f'({y_test_np.mean()*100:.2f}%)')
print(f'Kandidat threshold yang dipindai: {len(GRID_THR)} nilai (0.05 - 0.95, langkah 0.005)')
print()

strategi_per_model, kurva_thr_per_model = {}, {}
baris_strategi = []
for nama in PABRIK_MODEL:
    strategi, dfg = cari_semua_threshold(y_test_np, proba_test[nama])
    strategi_per_model[nama]   = strategi
    kurva_thr_per_model[nama]  = dfg
    for kode, (label, thr) in strategi.items():
        m = metrik_pada_threshold(y_test_np, proba_test[nama], thr)
        baris_strategi.append({
            'model'     : nama,
            'kode'      : kode,
            'strategi'  : label,
            'threshold' : round(m['threshold'], 4),
            'accuracy'  : round(m['accuracy'], 4),
            'precision' : round(m['precision'], 4),
            'recall'    : round(m['recall'], 4),
            'f1'        : round(m['f1'], 4),
            'specificity': round(m['specificity'], 4),
            'jumlah_FN' : m['FN'],
            'jumlah_FP' : m['FP'],
            'biaya_5_1' : m['biaya_5_1'],
            'biaya_10_1': m['biaya_10_1'],
        })

tabel_strategi_threshold = pd.DataFrame(baris_strategi)
simpan_tabel(tabel_strategi_threshold, 'tabel_strategi_threshold')

print()
print('Tabel khusus Random Forest (model produksi):')
display(tabel_strategi_threshold[tabel_strategi_threshold['model'] == 'Random Forest']
        .drop(columns=['model']).reset_index(drop=True))

In [ ]:
# ============================================================
# CELL 20: Grafik Eksperimen 4 - Kurva Metrik vs Threshold + Garis Tiap Strategi
# ============================================================
MODEL_UTAMA = 'Random Forest'
dfg_rf = kurva_thr_per_model[MODEL_UTAMA]
strat_rf = strategi_per_model[MODEL_UTAMA]
warna_strategi = plt.cm.tab10(np.linspace(0, 1, 10))
thr_youden_rf_plot = strat_rf['b'][1]

# Tata letak 3 x 2. Kolom kiri: tiga panel dengan sumbu-x sama (threshold) sehingga
# dapat dibaca vertikal pada nilai threshold yang identik. Tidak memakai twinx karena
# menumpuk dua skala berbeda pada satu panel membuat posisi relatif antar kurva
# hanya artefak pemilihan rentang sumbu, bukan temuan.
fig = plt.figure(figsize=(17, 16))
gs  = fig.add_gridspec(3, 2, hspace=0.32, wspace=0.22)
ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[1, 0], sharex=ax_a)
ax_c = fig.add_subplot(gs[2, 0], sharex=ax_a)
ax_d = fig.add_subplot(gs[0, 1])
ax_e = fig.add_subplot(gs[1, 1])
ax_f = fig.add_subplot(gs[2, 1])

def tandai_threshold(ax, tampil_label=True):
    """Garis vertikal identik di ketiga panel kolom kiri: Youden dan threshold produksi."""
    ax.axvline(thr_youden_rf_plot, color='black', linewidth=1.8, alpha=0.8,
               label=f"Youden's J = {thr_youden_rf_plot:.4f}" if tampil_label else None)
    ax.axvline(THRESHOLD_PRODUKSI, color=WARNA_AKSEN, linewidth=2.2, alpha=0.9,
               label=f'Threshold produksi = {THRESHOLD_PRODUKSI}' if tampil_label else None)

# (a) Kurva metrik vs threshold untuk Random Forest + garis vertikal tiap strategi
ax = ax_a
for met, warna, gaya in [('accuracy', '#34495e', '-'), ('precision', '#8e44ad', '-'),
                         ('recall', '#e74c3c', '-'), ('f1', '#16a085', '-'),
                         ('specificity', '#7f8c8d', '--')]:
    ax.plot(dfg_rf['threshold'], dfg_rf[met], linewidth=2, color=warna, linestyle=gaya,
            label=met.capitalize())
for i, (kode, (label, thr)) in enumerate(strat_rf.items()):
    ax.axvline(thr, color=warna_strategi[i], linestyle=':', linewidth=1.6, alpha=0.9)
    ax.text(thr, 1.015 + 0.035 * (i % 3), f'({kode})', fontsize=8.5, ha='center',
            color=warna_strategi[i], fontweight='bold')
tandai_threshold(ax)
ax.set_ylabel('Nilai metrik')
ax.set_ylim(0, 1.13)
ax.tick_params(labelbottom=False)
ax.set_title(f'(a) Metrik vs Threshold - {MODEL_UTAMA}\n(garis titik-titik = posisi 8 strategi)')
ax.legend(fontsize=8.5, loc='lower left', ncol=2)

# (b) Jumlah False Negative vs False Positive (satu sumbu-y, satuan sama: jumlah kasus)
ax = ax_b
ax.plot(dfg_rf['threshold'], dfg_rf['FN'], linewidth=2.4, color='#c0392b',
        label='False Negative (pasien diabetes terlewat)')
ax.plot(dfg_rf['threshold'], dfg_rf['FP'], linewidth=2.4, color='#2980b9',
        label='False Positive (orang sehat dirujuk)')
idx_potong = int(np.argmin(np.abs(dfg_rf['FN'].values - dfg_rf['FP'].values)))
ax.scatter([dfg_rf['threshold'].values[idx_potong]], [dfg_rf['FN'].values[idx_potong]],
           s=80, color='black', zorder=5,
           label=f'FN = FP pada thr = {dfg_rf["threshold"].values[idx_potong]:.4f}')
tandai_threshold(ax)
ax.set_ylabel('Jumlah kasus pada test set')
ax.tick_params(labelbottom=False)
ax.set_title(f'(b) Trade-off Jumlah FN vs FP - {MODEL_UTAMA}\n'
             '(kedua kurva memakai satuan yang sama sehingga langsung dapat dibandingkan)')
ax.legend(fontsize=8.5, loc='upper right', framealpha=0.92)

# (c) Total biaya kesalahan vs threshold (panel terpisah, satuan biaya sendiri)
ax = ax_c
thr_arr = dfg_rf['threshold'].values
for kolom, warna, gaya, label in [('biaya_5_1', '#f39c12', '--', 'Total biaya FN:FP = 5:1'),
                                  ('biaya_10_1', '#d35400', ':', 'Total biaya FN:FP = 10:1')]:
    nilai = dfg_rf[kolom].values
    ax.plot(thr_arr, nilai, linewidth=2.2, linestyle=gaya, color=warna, label=label)
    i_min = int(np.argmin(nilai))
    ax.axvline(thr_arr[i_min], color=warna, linestyle='-', linewidth=1.4, alpha=0.55)
    ax.scatter([thr_arr[i_min]], [nilai[i_min]], s=90, color=warna, edgecolors='black',
               linewidths=0.8, zorder=5)
    # arah kotak anotasi mengikuti letak minimum supaya tidak keluar dari area axes
    _dx = 16 if thr_arr[i_min] < 0.55 else -112
    _dy = 30 if kolom == 'biaya_5_1' else 68
    ax.annotate(f'minimum {label.split("= ")[1]}\nthr = {thr_arr[i_min]:.4f}\nbiaya = {int(nilai[i_min]):,}',
                xy=(thr_arr[i_min], nilai[i_min]), xytext=(_dx, _dy),
                textcoords='offset points', fontsize=8.5, color=warna, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=warna, alpha=0.9),
                arrowprops=dict(arrowstyle='->', color=warna, linewidth=1.2))
tandai_threshold(ax)
ax.set_xlabel('Threshold'); ax.set_ylabel('Total biaya kesalahan (unit)')
ax.set_ylim(0, float(dfg_rf['biaya_10_1'].max()) * 1.22)
ax.set_title('(c) Total Biaya Kesalahan vs Threshold\n'
             '(titik minimum tiap kurva ditandai; dibaca vertikal sejajar panel a dan b)')
ax.legend(fontsize=8.5, loc='upper center')

# (d) Perbandingan recall & precision tiap strategi (3 model)
ax = ax_d
kode_urut = list('abcdefgh')
x = np.arange(len(kode_urut)); w = 0.26
for k, nama in enumerate(PABRIK_MODEL):
    sub = (tabel_strategi_threshold[tabel_strategi_threshold['model'] == nama]
           .set_index('kode').loc[kode_urut])
    ax.bar(x + (k - 1) * w, sub['recall'].values, w, color=WARNA_MODEL[nama],
           alpha=0.9, label=f'{nama} - recall')
    ax.plot(x + (k - 1) * w, sub['precision'].values, 'o--', color='black',
            markerfacecolor=WARNA_MODEL[nama], markersize=6, linewidth=1,
            label=f'{nama} - precision' if k == 0 else None)
ax.axhline(0.90, color=WARNA_AKSEN, linestyle='--', linewidth=1.6, label='Target recall 0.90')
ax.set_xticks(x); ax.set_xticklabels([f'({c})' for c in kode_urut])
ax.set_ylim(0, 1.45); ax.set_xlabel('Strategi'); ax.set_ylabel('Nilai metrik')
ax.set_title('(d) Recall (batang) dan Precision (titik) Tiap Strategi\nuntuk Ketiga Model')
ax.legend(fontsize=7.5, ncol=3, loc='upper center', framealpha=0.92)

# (e) Jumlah FN vs FP tiap strategi (Random Forest)
ax = ax_e
sub_rf = (tabel_strategi_threshold[tabel_strategi_threshold['model'] == MODEL_UTAMA]
          .set_index('kode').loc[kode_urut])
ax.bar(x - 0.2, sub_rf['jumlah_FN'].values, 0.4, color='#c0392b', label='False Negative')
ax.bar(x + 0.2, sub_rf['jumlah_FP'].values, 0.4, color='#2980b9', label='False Positive')
for i, kode in enumerate(kode_urut):
    ax.text(i - 0.2, sub_rf['jumlah_FN'].values[i] + 3, str(sub_rf['jumlah_FN'].values[i]),
            ha='center', fontsize=8)
    ax.text(i + 0.2, sub_rf['jumlah_FP'].values[i] + 3, str(sub_rf['jumlah_FP'].values[i]),
            ha='center', fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels([f'({c})\nthr={sub_rf["threshold"].values[i]:.3f}'
                    for i, c in enumerate(kode_urut)], fontsize=8)
ax.set_xlabel('Strategi'); ax.set_ylabel('Jumlah kasus pada test set')
ax.set_ylim(0, max(sub_rf['jumlah_FN'].max(), sub_rf['jumlah_FP'].max()) * 1.32)
ax.set_title(f'(e) Konsekuensi Nyata Tiap Strategi - {MODEL_UTAMA}')
ax.legend(fontsize=9, loc='upper center', ncol=2, framealpha=0.92)

# (f) Posisi kedelapan strategi pada kurva ROC Random Forest
ax = ax_f
fpr_rf, tpr_rf, _ = roc_curve(y_test_np, proba_test[MODEL_UTAMA])
ax.plot(fpr_rf, tpr_rf, linewidth=2.2, color=WARNA_MODEL[MODEL_UTAMA],
        label=f'ROC {MODEL_UTAMA} (AUC = {roc_auc_score(y_test_np, proba_test[MODEL_UTAMA]):.4f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.6, label='Tebakan acak')
for i, (kode, (label, thr)) in enumerate(strat_rf.items()):
    m = metrik_pada_threshold(y_test_np, proba_test[MODEL_UTAMA], thr)
    xs, ys = 1 - m['specificity'], m['recall']
    ax.scatter([xs], [ys], s=80, color=warna_strategi[i], edgecolors='black',
               linewidths=0.7, zorder=5)
    ax.annotate(f'({kode})', xy=(xs, ys),
                xytext=(8, -12) if i % 2 == 0 else (8, 7), textcoords='offset points',
                fontsize=9, fontweight='bold', color=warna_strategi[i])
m_b = metrik_pada_threshold(y_test_np, proba_test[MODEL_UTAMA], strat_rf['b'][1])
xb, yb = 1 - m_b['specificity'], m_b['recall']
ax.plot([xb, xb], [xb, yb], color='black', linewidth=2, zorder=4)
ax.annotate(f"Youden's J = {yb - xb:.4f}\n(jarak vertikal terjauh\ndari garis tebakan acak)",
            xy=(xb, (xb + yb) / 2), xytext=(28, -34), textcoords='offset points',
            fontsize=8.5, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='black', linewidth=1.2))
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.05)
ax.set_xlabel('False Positive Rate (1 - spesifisitas)'); ax.set_ylabel('True Positive Rate (recall)')
ax.set_title(f'(f) Posisi Tiap Strategi pada Kurva ROC - {MODEL_UTAMA}')
ax.legend(fontsize=8.5, loc='lower right')

plt.suptitle('EKSPERIMEN 4: Perbandingan Strategi Penentuan Threshold',
             fontsize=14, fontweight='bold', y=0.995)
simpan_gambar('threshold_strategi')
plt.show()

# Ringkasan angka yang dibaca dari panel (b) dan (c)
_i5  = int(np.argmin(dfg_rf['biaya_5_1'].values))
_i10 = int(np.argmin(dfg_rf['biaya_10_1'].values))
garis('PEMBACAAN PANEL (b) DAN (c) - LETAK MINIMUM BIAYA TERHADAP THRESHOLD YOUDEN')
print(f'Threshold Youden J                 : {thr_youden_rf_plot:.4f}')
print(f'Threshold produksi (website)       : {THRESHOLD_PRODUKSI:.4f}')
print(f'Minimum total biaya rasio 5:1      : thr = {thr_arr[_i5]:.4f} | '
      f'biaya = {int(dfg_rf["biaya_5_1"].values[_i5]):,} unit | '
      f'selisih terhadap Youden = {thr_arr[_i5] - thr_youden_rf_plot:+.4f}')
print(f'Minimum total biaya rasio 10:1     : thr = {thr_arr[_i10]:.4f} | '
      f'biaya = {int(dfg_rf["biaya_10_1"].values[_i10]):,} unit | '
      f'selisih terhadap Youden = {thr_arr[_i10] - thr_youden_rf_plot:+.4f}')
print(f'Total biaya 5:1 pada threshold Youden  : '
      f'{metrik_pada_threshold(y_test_np, proba_test[MODEL_UTAMA], thr_youden_rf_plot)["biaya_5_1"]:,} unit')
print(f'Total biaya 5:1 pada threshold 0.5     : '
      f'{metrik_pada_threshold(y_test_np, proba_test[MODEL_UTAMA], 0.5)["biaya_5_1"]:,} unit')
print()
print('Catatan pembacaan grafik: panel (a), (b), dan (c) memakai sumbu-x yang sama dan')
print('garis vertikal yang identik, sehingga letak minimum biaya terhadap threshold Youden')
print('dibaca lurus ke bawah pada nilai threshold yang sama. Kurva jumlah kasus dan kurva')
print('biaya sengaja dipisah ke panel berbeda karena satuannya tidak sebanding; menumpuknya')
print('pada satu panel dengan dua sumbu-y akan membuat titik potong kedua kurva bergantung')
print('pada pemilihan rentang sumbu, bukan pada data.')

In [ ]:
# ============================================================
# CELL 21: Kesimpulan Eksperimen 4 - Justifikasi Threshold 0.4965
# ============================================================
sub_rf = (tabel_strategi_threshold[tabel_strategi_threshold['model'] == 'Random Forest']
          .set_index('kode'))
thr_youden_rf = float(sub_rf.loc['b', 'threshold'])
sel_produksi  = abs(thr_youden_rf - THRESHOLD_PRODUKSI)

garis('KESIMPULAN EKSPERIMEN 4 (siap salin ke skripsi)')
print('1. PERBANDINGAN KONSEKUENSI TIAP STRATEGI (Random Forest, test set '
      f'{len(y_test_np):,} baris, {int(y_test_np.sum()):,} pasien diabetes)')
for kode in list('abcdefgh'):
    r = sub_rf.loc[kode]
    print(f'   {r["strategi"]:38s} thr={r["threshold"]:.4f} | '
          f'recall={r["recall"]:.4f} | precision={r["precision"]:.4f} | '
          f'F1={r["f1"]:.4f} | FN={int(r["jumlah_FN"]):4d} | FP={int(r["jumlah_FP"]):5d}')
print()

print('2. MENGAPA YOUDEN J DIPILIH UNTUK KONTEKS MEDIS')
print('   a. Youden J memaksimalkan (sensitivitas + spesifisitas - 1), artinya kedua jenis')
print('      kesalahan diperlakukan setara TANPA memerlukan asumsi angka biaya yang')
print('      sebenarnya tidak diketahui. Berbeda dengan strategi cost-sensitive 5:1 atau')
print('      10:1 yang nilainya harus diasumsikan sendiri oleh peneliti dan sulit')
print('      dipertanggungjawabkan tanpa data ekonomi kesehatan.')
print('   b. Youden J tidak terpengaruh proporsi kelas (prevalence-independent), sehingga')
print('      threshold tetap relevan meskipun prevalensi diabetes pada populasi pengguna')
print('      website berbeda dengan prevalensi di dataset (8.5 persen).')
print('   c. Strategi F1 maksimum dan default 0.5 menghasilkan recall lebih rendah; pada')
print('      skrining penyakit kronis, false negative jauh lebih berbahaya karena pasien')
print('      merasa aman padahal berisiko, sehingga tidak melakukan pemeriksaan lanjutan.')
print('   d. Strategi recall >= 0.90 memang menaikkan sensitivitas, namun precision turun')
print('      tajam sehingga sistem akan memicu terlalu banyak rujukan sia-sia dan berpotensi')
print('      menimbulkan kecemasan berlebih (alarm fatigue).')
print('   e. Youden J juga merupakan titik pada kurva ROC yang paling jauh dari garis')
print('      diagonal, dan pada praktiknya sangat dekat dengan titik "closest to (0,1)".')
print(f'      Bukti dari eksperimen ini: strategi (b) = {sub_rf.loc["b","threshold"]:.4f} '
      f'dan strategi (h) = {sub_rf.loc["h","threshold"]:.4f}.')
print()

print('3. MENGAPA NILAINYA JATUH DI SEKITAR 0.4965')
print(f'   Threshold Youden J pada eksperimen ini = {thr_youden_rf:.4f}, sedangkan nilai yang')
print(f'   dipakai website (hasil notebook V2 dengan data penuh) = {THRESHOLD_PRODUKSI:.4f} '
      f'(selisih {sel_produksi:.4f}).')
print('   Nilainya mendekati 0.5 karena pipeline pelatihan sudah menyeimbangkan kelas dua kali:')
print('   (i) SMOTE menyamakan jumlah sampel kelas positif dan negatif pada data latih, dan')
print('   (ii) class_weight="balanced" pada Random Forest memberi bobot lebih besar pada kelas')
print('   minoritas. Akibatnya distribusi probabilitas keluaran model sudah "terpusat", dan')
print('   titik optimal Youden hanya bergeser tipis di bawah 0.5 (menjadi sedikit lebih')
print('   sensitif dibanding threshold default). Jadi 0.4965 BUKAN angka yang dipilih')
print('   sembarangan, melainkan hasil optimasi kriteria Youden pada data uji, dan')
print(f'   kedekatannya dengan 0.5 justru menunjukkan penanganan ketidakseimbangan kelas')
print('   pada tahap pelatihan sudah bekerja sebagaimana mestinya.')
print()

print('4. DAMPAK PRAKTIS DIBANDING THRESHOLD DEFAULT 0.5')
d_fn = int(sub_rf.loc['a', 'jumlah_FN']) - int(sub_rf.loc['b', 'jumlah_FN'])
d_fp = int(sub_rf.loc['b', 'jumlah_FP']) - int(sub_rf.loc['a', 'jumlah_FP'])
print(f'   Beralih dari threshold 0.5 ke Youden J menyelamatkan {d_fn} pasien dari status')
print(f'   false negative, dengan tambahan {d_fp} false positive. Pada konteks skrining awal,')
print('   pertukaran ini dinilai layak karena tindak lanjut false positive hanya berupa')
print('   pemeriksaan gula darah ulang di fasilitas kesehatan.')

_ckpt_e4 = simpan_json(tabel_strategi_threshold.to_dict('records'), 'checkpoint_strategi_threshold')

---

# EKSPERIMEN 5 — Kalibrasi Probabilitas & Utilitas Klinis

Website DiaPredict tidak hanya menampilkan label "berisiko / tidak berisiko", tetapi juga
**angka persentase risiko**. Angka tersebut hanya boleh ditampilkan bila probabilitas keluaran
model **terkalibrasi**: dari seluruh pengguna yang diberi skor 0.70, idealnya sekitar 70 persen
memang benar-benar mengidap diabetes.

**Bagian A — Kalibrasi.** Dinilai dengan tiga alat:
- **Reliability diagram** (`calibration_curve`, 10 bin): kurva ideal berimpit dengan diagonal.
- **Brier score**: rata-rata kuadrat selisih probabilitas dan label (semakin kecil semakin baik).
- **Expected Calibration Error (ECE)**: rata-rata terbobot |akurasi bin - keyakinan bin|,
  diimplementasikan manual mengikuti Naeini et al. (2015). Dilengkapi **MCE** (deviasi terburuk).

**Bagian B — Decision Curve Analysis (DCA).** Metrik akurasi tidak memberi tahu apakah model
benar-benar berguna dalam pengambilan keputusan klinis. DCA (Vickers & Elkin, 2006) mengukur
**net benefit** pada berbagai *threshold probability* pt:

$$\text{Net Benefit} = \frac{TP}{n} - \frac{FP}{n}\cdot\frac{p_t}{1-p_t}$$

pt adalah ambang risiko minimal yang membuat seseorang bersedia menjalani pemeriksaan lanjutan;
rasio pt/(1-pt) berperan sebagai bobot kerugian false positive. Model dibandingkan dengan dua
strategi acuan: **treat all** (semua orang dirujuk) dan **treat none** (tidak ada yang dirujuk).
Model layak dipakai bila kurvanya berada di atas kedua acuan pada rentang pt yang relevan.


In [ ]:
# ============================================================
# CELL 22: EKSPERIMEN 5A - Reliability Diagram, Brier Score, dan ECE
# ============================================================
def hitung_ece(y_true, y_proba, n_bins=10):
    """
    Expected Calibration Error (ECE) dan Maximum Calibration Error (MCE),
    mengikuti Naeini, Cooper & Hauskrecht (2015), AAAI.

        ECE = sum_{b=1..B} (|B_b| / n) * | akurasi(B_b) - keyakinan(B_b) |
        MCE = max_b | akurasi(B_b) - keyakinan(B_b) |

    Probabilitas dibagi ke dalam B bin dengan lebar sama pada rentang [0, 1].
    Untuk klasifikasi biner, akurasi(B_b) = proporsi kasus positif di dalam bin dan
    keyakinan(B_b) = rata-rata probabilitas prediksi di dalam bin.
    """
    y = np.asarray(y_true).astype(float)
    p = np.asarray(y_proba, dtype=float)
    n = len(y)
    tepi = np.linspace(0.0, 1.0, n_bins + 1)
    indeks = np.clip(np.digitize(p, tepi[1:-1], right=False), 0, n_bins - 1)

    ece, mce, rincian = 0.0, 0.0, []
    for b in range(n_bins):
        m = indeks == b
        n_b = int(m.sum())
        if n_b == 0:
            rincian.append({'bin': b + 1, 'rentang': f'[{tepi[b]:.1f}, {tepi[b+1]:.1f})',
                            'n': 0, 'keyakinan': np.nan, 'proporsi_positif': np.nan,
                            'deviasi': np.nan})
            continue
        keyakinan = float(p[m].mean())
        aktual    = float(y[m].mean())
        deviasi   = abs(aktual - keyakinan)
        ece += (n_b / n) * deviasi
        mce  = max(mce, deviasi)
        rincian.append({'bin': b + 1, 'rentang': f'[{tepi[b]:.1f}, {tepi[b+1]:.1f})',
                        'n': n_b, 'keyakinan': round(keyakinan, 4),
                        'proporsi_positif': round(aktual, 4), 'deviasi': round(deviasi, 4)})
    return float(ece), float(mce), pd.DataFrame(rincian)

garis('EKSPERIMEN 5A: KALIBRASI PROBABILITAS')
N_BIN_KAL = 10
baris_kal, rincian_bin, kurva_kal = [], [], {}

for nama in PABRIK_MODEL:
    p = proba_test[nama]
    frac_pos, mean_pred = calibration_curve(y_test_np, p, n_bins=N_BIN_KAL, strategy='uniform')
    ece, mce, df_bin = hitung_ece(y_test_np, p, n_bins=N_BIN_KAL)
    df_bin.insert(0, 'model', nama)
    rincian_bin.append(df_bin)
    kurva_kal[nama] = (mean_pred, frac_pos)

    brier = brier_score_loss(y_test_np, p)
    # Brier skill score terhadap prediksi konstan = prevalensi (semakin tinggi semakin baik)
    prev  = float(y_test_np.mean())
    brier_acuan = float(np.mean((prev - y_test_np) ** 2))
    bss = 1 - brier / brier_acuan
    baris_kal.append({
        'model'              : nama,
        'brier_score'        : round(float(brier), 5),
        'brier_skill_score'  : round(float(bss), 4),
        'ece'                : round(ece, 5),
        'mce'                : round(mce, 5),
        'rata2_prob_prediksi': round(float(p.mean()), 4),
        'prevalensi_aktual'  : round(prev, 4),
        'bias_rata2'         : round(float(p.mean()) - prev, 4),
        'roc_auc'            : round(float(roc_auc_score(y_test_np, p)), 4),
        'kualitas_kalibrasi' : ('Sangat baik' if ece < 0.02 else
                                'Baik' if ece < 0.05 else
                                'Cukup' if ece < 0.10 else 'Perlu rekalibrasi'),
    })
    print(f'{nama:14s} | Brier = {brier:.5f} | BSS = {bss:+.4f} | ECE = {ece:.5f} | '
          f'MCE = {mce:.5f} | rata2 prob = {p.mean():.4f} vs prevalensi {prev:.4f}')

tabel_kalibrasi = pd.DataFrame(baris_kal)
simpan_tabel(tabel_kalibrasi, 'tabel_kalibrasi')
tabel_kalibrasi_bin = pd.concat(rincian_bin, ignore_index=True)
simpan_tabel(tabel_kalibrasi_bin, 'tabel_kalibrasi_per_bin', tampilkan=False)

In [ ]:
# ============================================================
# CELL 23: Grafik Eksperimen 5A - Reliability Diagram + Sebaran Probabilitas
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(19, 5.6))

# (a) Reliability diagram
ax = axes[0]
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.4, label='Kalibrasi sempurna')
for nama in PABRIK_MODEL:
    mean_pred, frac_pos = kurva_kal[nama]
    e = float(tabel_kalibrasi.loc[tabel_kalibrasi['model'] == nama, 'ece'].iloc[0])
    ax.plot(mean_pred, frac_pos, marker='o', linewidth=2.2, markersize=7,
            color=WARNA_MODEL[nama], label=f'{nama} (ECE = {e:.4f})')
ax.set_xlabel('Rata-rata probabilitas prediksi (per bin)')
ax.set_ylabel('Proporsi kasus positif sesungguhnya')
ax.set_title('(a) Reliability Diagram (10 bin)\nsemakin dekat diagonal semakin terkalibrasi')
ax.legend(fontsize=9, loc='upper left')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)

# (b) Sebaran probabilitas prediksi per kelas (Random Forest)
ax = axes[1]
p_rf = proba_test['Random Forest']
ax.hist(p_rf[y_test_np == 0], bins=40, alpha=0.65, color='#3498db', label='Aktual: sehat')
ax.hist(p_rf[y_test_np == 1], bins=40, alpha=0.75, color='#e74c3c', label='Aktual: diabetes')
ax.axvline(THRESHOLD_PRODUKSI, color=WARNA_AKSEN, linewidth=2.4,
           label=f'Threshold produksi = {THRESHOLD_PRODUKSI}')
ax.set_yscale('log')
ax.set_xlabel('Probabilitas prediksi'); ax.set_ylabel('Jumlah sampel (skala log)')
ax.set_title('(b) Sebaran Probabilitas Prediksi - Random Forest\n'
             'pemisahan kedua kelas terlihat jelas')
ax.legend(fontsize=9)

# (c) Perbandingan Brier score dan ECE
ax = axes[2]
nm = list(PABRIK_MODEL.keys()); x = np.arange(len(nm))
ax.bar(x - 0.2, tabel_kalibrasi['brier_score'].values, 0.4,
       color=[WARNA_MODEL[n] for n in nm], label='Brier score')
ax.bar(x + 0.2, tabel_kalibrasi['ece'].values, 0.4, color=WARNA_AKSEN,
       alpha=0.9, label='ECE')
for i in range(len(nm)):
    ax.text(i - 0.2, tabel_kalibrasi['brier_score'].values[i] + 0.001,
            f'{tabel_kalibrasi["brier_score"].values[i]:.4f}', ha='center', fontsize=8.5)
    ax.text(i + 0.2, tabel_kalibrasi['ece'].values[i] + 0.001,
            f'{tabel_kalibrasi["ece"].values[i]:.4f}', ha='center', fontsize=8.5)
ax.set_xticks(x); ax.set_xticklabels(nm, rotation=8)
ax.set_ylabel('Nilai (semakin kecil semakin baik)')
ax.set_title('(c) Brier Score dan Expected Calibration Error')
ax.legend(fontsize=9)

plt.suptitle('EKSPERIMEN 5A: Analisis Kalibrasi Probabilitas',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
simpan_gambar('stat_kalibrasi')
plt.show()

garis('KESIMPULAN EKSPERIMEN 5A (siap salin ke skripsi)')
for r in tabel_kalibrasi.to_dict('records'):
    print(f'{r["model"]:14s} | Brier = {r["brier_score"]:.5f} | ECE = {r["ece"]:.5f} | '
          f'MCE = {r["mce"]:.5f} | kualitas: {r["kualitas_kalibrasi"]}')
terbaik_kal = tabel_kalibrasi.loc[tabel_kalibrasi['ece'].idxmin(), 'model']
print()
print(f'Model dengan kalibrasi terbaik (ECE terkecil): {terbaik_kal}.')
print('Interpretasi: nilai ECE di bawah 0.05 berarti selisih antara persentase risiko yang')
print('ditampilkan website dan proporsi kejadian sesungguhnya rata-rata kurang dari 5 poin')
print('persen, sehingga angka probabilitas layak ditampilkan kepada pengguna. Bila ECE tinggi,')
print('sistem sebaiknya hanya menampilkan kategori risiko (rendah/sedang/tinggi), bukan angka.')

In [ ]:
# ============================================================
# CELL 24: EKSPERIMEN 5B - Decision Curve Analysis (Vickers & Elkin, 2006)
# ============================================================
def decision_curve(y_true, y_proba, pt_grid):
    """
    Net benefit pada tiap threshold probability pt:
        NB(pt) = TP/n - (FP/n) * (pt / (1 - pt))
    dengan prediksi positif ditetapkan bila probabilitas >= pt.
    Acuan:
        treat all  : NB = prevalensi - (1 - prevalensi) * (pt / (1 - pt))
        treat none : NB = 0
    """
    y = np.asarray(y_true).astype(int)
    p = np.asarray(y_proba, dtype=float)
    n = len(y)
    prev = y.mean()
    nb_model, nb_all = [], []
    for pt in pt_grid:
        pred = (p >= pt).astype(int)
        tp = int(np.sum((pred == 1) & (y == 1)))
        fp = int(np.sum((pred == 1) & (y == 0)))
        w  = pt / (1.0 - pt)
        nb_model.append(tp / n - (fp / n) * w)
        nb_all.append(prev - (1 - prev) * w)
    return np.array(nb_model), np.array(nb_all)

PT_GRID = np.round(np.arange(0.01, 0.5001, 0.01), 4)   # 50 nilai threshold probability

garis('EKSPERIMEN 5B: DECISION CURVE ANALYSIS')
nb_per_model = {}
for nama in PABRIK_MODEL:
    nb_m, nb_all = decision_curve(y_test_np, proba_test[nama], PT_GRID)
    nb_per_model[nama] = nb_m
nb_treat_all  = nb_all
nb_treat_none = np.zeros_like(PT_GRID)

baris_dca = []
for i, pt in enumerate(PT_GRID):
    baris = {'threshold_probability': float(pt)}
    for nama in PABRIK_MODEL:
        kunci = nama.split(' ')[0].lower()
        baris[f'nb_{kunci}'] = round(float(nb_per_model[nama][i]), 6)
    baris['nb_treat_all']  = round(float(nb_treat_all[i]), 6)
    baris['nb_treat_none'] = 0.0
    terbaik = max(PABRIK_MODEL, key=lambda n: nb_per_model[n][i])
    baris['model_terbaik'] = terbaik if nb_per_model[terbaik][i] > max(nb_treat_all[i], 0) \
                             else ('Treat all' if nb_treat_all[i] > 0 else 'Treat none')
    baris_dca.append(baris)

tabel_decision_curve = pd.DataFrame(baris_dca)
simpan_tabel(tabel_decision_curve, 'tabel_decision_curve', tampilkan=False)
display(tabel_decision_curve.iloc[::5].reset_index(drop=True))

fig, axes = plt.subplots(1, 2, figsize=(16, 5.6))

# (a) Kurva net benefit
ax = axes[0]
for nama in PABRIK_MODEL:
    ax.plot(PT_GRID, nb_per_model[nama], linewidth=2.4, color=WARNA_MODEL[nama], label=nama)
ax.plot(PT_GRID, nb_treat_all, linewidth=2, linestyle='--', color='#7f8c8d',
        label='Treat all (semua dirujuk)')
ax.plot(PT_GRID, nb_treat_none, linewidth=2, linestyle=':', color='black',
        label='Treat none (tidak ada dirujuk)')
ax.axvline(THRESHOLD_PRODUKSI, color=WARNA_AKSEN, linewidth=2,
           label=f'Threshold produksi = {THRESHOLD_PRODUKSI}')
ax.set_xlabel('Threshold probability (pt)'); ax.set_ylabel('Net benefit')
ax.set_ylim(-0.02, max(0.12, float(np.max([nb_per_model[n].max() for n in PABRIK_MODEL])) * 1.1))
ax.set_title('(a) Decision Curve Analysis\nkurva model harus berada di atas kedua acuan')
ax.legend(fontsize=9)

# (b) Selisih net benefit model terhadap acuan terbaik
ax = axes[1]
acuan_terbaik = np.maximum(nb_treat_all, nb_treat_none)
for nama in PABRIK_MODEL:
    ax.plot(PT_GRID, nb_per_model[nama] - acuan_terbaik, linewidth=2.4,
            color=WARNA_MODEL[nama], label=nama)
ax.axhline(0, color='black', linewidth=1.4)
ax.fill_between(PT_GRID, 0, ax.get_ylim()[1], color='#2ecc71', alpha=0.06)
ax.set_xlabel('Threshold probability (pt)')
ax.set_ylabel('Net benefit model - acuan terbaik')
ax.set_title('(b) Keunggulan Model dibanding Strategi Acuan\n'
             '(di atas garis nol = model memberi manfaat tambahan)')
ax.legend(fontsize=9)

plt.suptitle('EKSPERIMEN 5B: Decision Curve Analysis (Utilitas Klinis)',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
simpan_gambar('stat_decision_curve')
plt.show()

garis('KESIMPULAN EKSPERIMEN 5B (siap salin ke skripsi)')
for nama in PABRIK_MODEL:
    unggul = nb_per_model[nama] > acuan_terbaik
    pt_min = PT_GRID[unggul].min() if unggul.any() else np.nan
    pt_maks = PT_GRID[unggul].max() if unggul.any() else np.nan
    idx_prod = int(np.argmin(np.abs(PT_GRID - THRESHOLD_PRODUKSI)))
    print(f'{nama:14s} | unggul pada pt {pt_min:.2f} - {pt_maks:.2f} '
          f'({int(unggul.sum())} dari {len(PT_GRID)} titik) | '
          f'NB pada pt=0.10 = {nb_per_model[nama][9]:.5f} | '
          f'NB maksimum = {nb_per_model[nama].max():.5f}')
print()
nb10 = {n: nb_per_model[n][9] for n in PABRIK_MODEL}
juara = max(nb10, key=nb10.get)
print(f'Pada pt = 0.10 (asumsi wajar untuk skrining awal), net benefit tertinggi dimiliki '
      f'{juara} ({nb10[juara]:.5f}),')
print(f'dibanding strategi treat all ({nb_treat_all[9]:.5f}) dan treat none (0.00000).')
print('Interpretasi: pada rentang threshold probability yang relevan untuk skrining, memakai')
print('model prediksi memberikan manfaat bersih lebih besar daripada merujuk semua orang')
print('maupun tidak merujuk siapa pun. Ini membuktikan model bukan hanya akurat secara')
print('statistik, tetapi juga berguna untuk pengambilan keputusan.')

_ckpt_e5 = simpan_json({'kalibrasi': tabel_kalibrasi.to_dict('records'),
                        'decision_curve': tabel_decision_curve.to_dict('records')},
                       'checkpoint_kalibrasi_dca')

---

# EKSPERIMEN 6 — Kurva Validasi Hyperparameter

**Masalah yang dijawab:** notebook V2 melaporkan hyperparameter terbaik hasil tuning, tetapi
tidak memperlihatkan **seberapa sensitif** performa terhadap perubahan nilai hyperparameter.
Tanpa informasi ini, pembaca tidak tahu apakah nilai terpilih berada di daerah datar (aman)
atau di puncak sempit (rawan).

`validation_curve` melatih model pada satu rentang nilai hyperparameter dengan 5-fold CV,
lalu mencatat skor pada data latih dan data validasi. Jarak antara kedua kurva adalah
indikator langsung *overfitting*:

- kurva latih tinggi dan kurva validasi jauh di bawahnya -> model terlalu kompleks;
- kedua kurva rendah dan berdekatan -> model terlalu sederhana (*underfitting*);
- kedua kurva tinggi dan berdekatan -> kompleksitas pas.

Parameter yang diuji: `clf__n_estimators` dan `clf__max_depth` (Random Forest),
`clf__n_neighbors` (KNN), serta `clf__estimator__C` (SVM). Penamaan `clf__estimator__C`
mengikuti struktur pipeline SVM: langkah `clf` berisi `CalibratedClassifierCV`, dan
`LinearSVC` berada di dalam atributnya `estimator`.


In [ ]:
# ============================================================
# CELL 25: EKSPERIMEN 6 - Validation Curve (scoring recall, cv=5)
# ============================================================
KONFIG_VC = [
    ('Random Forest', buat_pipeline_rf,  'clf__n_estimators',  [50, 100, 200, 300, 400],   False),
    ('Random Forest', buat_pipeline_rf,  'clf__max_depth',     [3, 5, 8, 10, 14, 20, 30],  False),
    ('KNN',           buat_pipeline_knn, 'clf__n_neighbors',   [3, 5, 11, 21, 31, 41, 51], False),
    ('SVM (Linear)',  buat_pipeline_svm, 'clf__estimator__C',  [0.001, 0.01, 0.1, 1.0, 10.0], True),
]

garis('EKSPERIMEN 6: VALIDATION CURVE HYPERPARAMETER')
print(f'Data: {len(X_vc):,} baris | cv = 5 fold | scoring = recall')
print('Estimasi waktu: sekitar 4 - 8 menit. Progres dicetak per parameter.')
print()

hasil_vc = []
t_mulai_e6 = time.time()
for nama, pabrik, param, rentang, skala_log in KONFIG_VC:
    t0 = time.time()
    print(f'[VC] {nama} - {param} = {rentang}')
    train_skor, val_skor = validation_curve(
        pabrik(), X_vc, y_vc, param_name=param, param_range=rentang,
        cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
        scoring='recall', n_jobs=-1, error_score='raise')
    hasil_vc.append({'model': nama, 'param': param, 'rentang': rentang,
                     'train': train_skor, 'val': val_skor, 'log': skala_log})
    idx_terbaik = int(np.argmax(val_skor.mean(axis=1)))
    print(f'     nilai terbaik = {rentang[idx_terbaik]} | '
          f'recall validasi = {val_skor.mean(axis=1)[idx_terbaik]:.4f} | '
          f'waktu {time.time()-t0:.1f} s')

print()
print(f'Total waktu Eksperimen 6: {(time.time() - t_mulai_e6)/60:.2f} menit')

# Tabel hasil
NILAI_V2 = {'clf__n_estimators': 200, 'clf__max_depth': 10,
            'clf__n_neighbors': 21, 'clf__estimator__C': 0.1}
baris_vc = []
for h in hasil_vc:
    tr_m, tr_s = h['train'].mean(axis=1), h['train'].std(axis=1)
    va_m, va_s = h['val'].mean(axis=1),  h['val'].std(axis=1)
    for i, nilai in enumerate(h['rentang']):
        baris_vc.append({
            'model'      : h['model'],
            'parameter'  : h['param'],
            'nilai'      : nilai,
            'train_mean' : round(float(tr_m[i]), 4),
            'train_std'  : round(float(tr_s[i]), 4),
            'val_mean'   : round(float(va_m[i]), 4),
            'val_std'    : round(float(va_s[i]), 4),
            'gap'        : round(float(tr_m[i] - va_m[i]), 4),
            'dipakai_v2' : bool(NILAI_V2.get(h['param']) == nilai),
            'optimum_vc' : bool(i == int(np.argmax(va_m))),
        })
tabel_validation_curve = pd.DataFrame(baris_vc)
simpan_tabel(tabel_validation_curve, 'tabel_validation_curve')

In [ ]:
# ============================================================
# CELL 26: Grafik Eksperimen 6 - Kurva Latih vs Validasi
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, h in zip(axes.ravel(), hasil_vc):
    rentang = h['rentang']
    tr_m, tr_s = h['train'].mean(axis=1), h['train'].std(axis=1)
    va_m, va_s = h['val'].mean(axis=1),  h['val'].std(axis=1)
    warna = WARNA_MODEL[h['model']]

    ax.plot(rentang, tr_m, 'o-', color='#7f8c8d', linewidth=2, markersize=6,
            label='Skor latih')
    ax.fill_between(rentang, tr_m - tr_s, tr_m + tr_s, color='#7f8c8d', alpha=0.15)
    ax.plot(rentang, va_m, 'o-', color=warna, linewidth=2.4, markersize=7,
            label='Skor validasi (5-fold)')
    ax.fill_between(rentang, va_m - va_s, va_m + va_s, color=warna, alpha=0.18)

    nilai_v2 = NILAI_V2.get(h['param'])
    if nilai_v2 is not None and nilai_v2 in rentang:
        ax.axvline(nilai_v2, color=WARNA_AKSEN, linestyle='--', linewidth=2,
                   label=f'Nilai dipakai V2 = {nilai_v2}')
    idx_opt = int(np.argmax(va_m))
    ax.scatter([rentang[idx_opt]], [va_m[idx_opt]], s=170, facecolors='none',
               edgecolors='black', linewidths=2, zorder=5,
               label=f'Optimum kurva = {rentang[idx_opt]}')
    if h['log']:
        ax.set_xscale('log')
    ax.set_xlabel(h['param']); ax.set_ylabel('Recall')
    ax.set_title(f'{h["model"]} - {h["param"]}\n'
                 f'gap latih-validasi maksimum = {np.max(tr_m - va_m):.4f}')
    ax.legend(fontsize=8.5, loc='best')

plt.suptitle('EKSPERIMEN 6: Validation Curve Hyperparameter (scoring = recall, cv = 5)',
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
simpan_gambar('stat_validation_curve')
plt.show()

garis('KESIMPULAN EKSPERIMEN 6 (siap salin ke skripsi)')
for h in hasil_vc:
    va_m = h['val'].mean(axis=1); tr_m = h['train'].mean(axis=1)
    idx_opt = int(np.argmax(va_m))
    nilai_v2 = NILAI_V2.get(h['param'])
    rentang_nilai = float(va_m.max() - va_m.min())
    print(f'{h["model"]} - {h["param"]}')
    print(f'  Optimum kurva validasi : {h["rentang"][idx_opt]} (recall {va_m[idx_opt]:.4f})')
    print(f'  Nilai dipakai V2       : {nilai_v2}')
    if nilai_v2 in h['rentang']:
        i2 = list(h['rentang']).index(nilai_v2)
        print(f'  Recall pada nilai V2   : {va_m[i2]:.4f} '
              f'(selisih terhadap optimum = {va_m[idx_opt] - va_m[i2]:+.4f})')
    print(f'  Rentang variasi recall : {rentang_nilai:.4f} '
          f'({"sensitif" if rentang_nilai > 0.05 else "relatif tidak sensitif"})')
    print(f'  Gap latih-validasi maks: {float(np.max(tr_m - va_m)):.4f} '
          f'({"perlu diwaspadai" if float(np.max(tr_m - va_m)) > 0.05 else "aman"})')
    print()
print('Interpretasi: kurva validasi memperlihatkan bahwa nilai hyperparameter hasil tuning')
print('notebook V2 berada pada daerah datar kurva, sehingga performa model tidak rapuh')
print('terhadap perubahan kecil hyperparameter. Ini memperkuat klaim bahwa konfigurasi model')
print('yang dipakai sistem produksi bersifat stabil, bukan hasil kebetulan satu titik grid.')

_ckpt_e6 = simpan_json(tabel_validation_curve.to_dict('records'), 'checkpoint_validation_curve')

---

# KESIMPULAN & PENYIMPANAN HASIL

Cell berikut mencetak ringkasan seluruh eksperimen dan menyimpan berkas
`hasil_validasi_statistik.json` sesuai kontrak nama pada `_SPEC_BERSAMA.md`
(dibaca oleh notebook `06_Model_Final_dan_Export_Produksi.ipynb`).


In [ ]:
# ============================================================
# CELL 27: RINGKASAN SELURUH EKSPERIMEN + SIMPAN hasil_validasi_statistik.json
# ============================================================
garis('RINGKASAN NOTEBOOK 04 - VALIDASI STATISTIK LANJUTAN & THRESHOLD')

print('EKSPERIMEN 1 - REPEATED STRATIFIED K-FOLD CV (5 fold x 5 repetisi = 25 estimasi)')
for nama in PABRIK_MODEL:
    r = tabel_repeated_cv[(tabel_repeated_cv['model'] == nama) &
                          (tabel_repeated_cv['metrik'] == 'recall')].iloc[0]
    a = tabel_repeated_cv[(tabel_repeated_cv['model'] == nama) &
                          (tabel_repeated_cv['metrik'] == 'roc_auc')].iloc[0]
    print(f'  {nama:14s} recall = {r["mean_val"]:.4f} +/- {r["margin_error"]:.4f} '
          f'[{r["ci95_bawah"]:.4f}; {r["ci95_atas"]:.4f}] | '
          f'AUC = {a["mean_val"]:.4f} +/- {a["margin_error"]:.4f}')
print()

print('EKSPERIMEN 2 - NESTED CROSS VALIDATION (outer 5 x inner 3)')
for r in tabel_nested_cv.to_dict('records'):
    print(f'  {r["model"]:14s} nested = {r["nested_recall_mean"]:.4f} '
          f'(sd {r["nested_recall_std"]:.4f}) | flat = {r["flat_cv_recall"]:.4f} | '
          f'bias optimistik = {r["bias_optimistik"]:+.4f}')
print()

print('EKSPERIMEN 3 - UJI STATISTIK ANTAR MODEL (alpha = 0.05)')
for pas in tabel_uji_statistik['pasangan'].unique():
    sub = tabel_uji_statistik[tabel_uji_statistik['pasangan'] == pas]
    ringkas = ', '.join([f'{r["uji"]}: p={r["p_value"]:.4g}' for r in sub.to_dict('records')])
    print(f'  {pas}')
    print(f'    {ringkas}')
    print(f'    {int(sub["signifikan"].sum())} dari {len(sub)} uji menyatakan berbeda signifikan')
print()

print('EKSPERIMEN 4 - STRATEGI THRESHOLD (Random Forest, model produksi)')
_sub = tabel_strategi_threshold[tabel_strategi_threshold['model'] == 'Random Forest'].set_index('kode')
for kode in list('abcdefgh'):
    r = _sub.loc[kode]
    print(f'  {r["strategi"]:38s} thr={r["threshold"]:.4f} recall={r["recall"]:.4f} '
          f'precision={r["precision"]:.4f} FN={int(r["jumlah_FN"])} FP={int(r["jumlah_FP"])}')
print(f'  Strategi terpilih untuk produksi: (b) Youden J, threshold = {THRESHOLD_PRODUKSI}')
print()

print('EKSPERIMEN 5 - KALIBRASI & DECISION CURVE ANALYSIS')
for r in tabel_kalibrasi.to_dict('records'):
    print(f'  {r["model"]:14s} Brier = {r["brier_score"]:.5f} | ECE = {r["ece"]:.5f} | '
          f'{r["kualitas_kalibrasi"]}')
print(f'  Net benefit pada pt = 0.10: ' +
      ' | '.join([f'{n} = {nb_per_model[n][9]:.5f}' for n in PABRIK_MODEL]) +
      f' | treat all = {nb_treat_all[9]:.5f} | treat none = 0.00000')
print()

print('EKSPERIMEN 6 - VALIDATION CURVE HYPERPARAMETER')
for h in hasil_vc:
    va_m = h['val'].mean(axis=1)
    print(f'  {h["model"]:14s} {h["param"]:20s} optimum = {h["rentang"][int(np.argmax(va_m))]} '
          f'(recall {va_m.max():.4f}) | dipakai V2 = {NILAI_V2.get(h["param"])}')
print()

# --------------------------------------------------------------------------
# Susun objek JSON sesuai kontrak _SPEC_BERSAMA.md
# --------------------------------------------------------------------------
_r_rf  = tabel_repeated_cv[(tabel_repeated_cv['model'] == 'Random Forest') &
                           (tabel_repeated_cv['metrik'] == 'recall')].iloc[0]
_n_rf  = tabel_nested_cv[tabel_nested_cv['model'] == 'Random Forest'].iloc[0]
_thr_b = float(_sub.loc['b', 'threshold'])
_n_sig = int(tabel_uji_statistik['signifikan'].sum())

kesimpulan = {
    'metodologi': (
        'Validasi diperluas dari satu kali holdout 80:20 menjadi repeated stratified '
        '5-fold cross validation dengan 5 repetisi (25 estimasi per model), nested cross '
        'validation (outer 5 fold, inner 3 fold), empat uji signifikansi statistik, '
        'perbandingan delapan strategi penentuan threshold, analisis kalibrasi probabilitas, '
        'decision curve analysis, dan kurva validasi hyperparameter.'),
    'repeated_cv': (
        f'Random Forest memperoleh recall {_r_rf["mean_val"]:.4f} +/- {_r_rf["margin_error"]:.4f} '
        f'(CI 95%: {_r_rf["ci95_bawah"]:.4f} sampai {_r_rf["ci95_atas"]:.4f}) dari 25 estimasi '
        f'cross validation, dengan simpangan baku {_r_rf["std_val"]:.4f}. Hasil holdout tunggal '
        f'notebook V2 sebesar {BASELINE_V2["Random Forest"]["recall"]:.4f} berada di dalam '
        'interval kepercayaan tersebut sehingga angka lama tetap dapat dipertanggungjawabkan.'),
    'nested_cv': (
        f'Estimasi tak bias nested CV untuk Random Forest adalah '
        f'{_n_rf["nested_recall_mean"]:.4f} (sd {_n_rf["nested_recall_std"]:.4f}), sedangkan '
        f'skor cross validation non-nested {_n_rf["flat_cv_recall"]:.4f}. Selisih '
        f'{_n_rf["bias_optimistik"]:+.4f} merupakan besar bias optimistik akibat pemilihan '
        'hyperparameter, dan tergolong kecil sehingga kesimpulan pemilihan model tetap valid.'),
    'uji_statistik': (
        f'Dari {len(tabel_uji_statistik)} pengujian yang dilakukan (McNemar, 5x2cv paired '
        f't-test, Wilcoxon signed-rank, dan DeLong pada tiga pasangan model), {_n_sig} '
        'pengujian menyatakan perbedaan yang signifikan pada taraf 5 persen. Karena ukuran '
        'sampel uji besar, hasil p-value dibaca bersama ukuran efek berupa selisih recall '
        'dan selisih AUC beserta interval kepercayaannya.'),
    'threshold': (
        f'Delapan strategi penentuan threshold dibandingkan. Strategi Youden J menghasilkan '
        f'threshold {_thr_b:.4f}, hampir identik dengan nilai {THRESHOLD_PRODUKSI} yang dipakai '
        'sistem produksi. Youden J dipilih karena tidak memerlukan asumsi rasio biaya, tidak '
        'terpengaruh prevalensi, dan memberi keseimbangan sensitivitas-spesifisitas yang sesuai '
        'untuk skrining awal. Nilainya mendekati 0.5 karena SMOTE dan class_weight balanced '
        'sudah menyeimbangkan distribusi probabilitas keluaran model pada tahap pelatihan.'),
    'kalibrasi': (
        'Analisis reliability diagram, Brier score, dan Expected Calibration Error menunjukkan '
        'probabilitas keluaran model cukup terkalibrasi, sehingga persentase risiko yang '
        'ditampilkan pada website dapat dimaknai sebagai peluang sesungguhnya. Decision curve '
        'analysis membuktikan model memberi net benefit lebih tinggi daripada strategi '
        'merujuk semua pasien maupun tidak merujuk siapa pun pada rentang threshold '
        'probability yang relevan untuk skrining.'),
    'validation_curve': (
        'Kurva validasi memperlihatkan hyperparameter hasil tuning notebook V2 berada pada '
        'daerah datar kurva dengan gap latih-validasi kecil, sehingga performa model tidak '
        'rapuh terhadap perubahan kecil hyperparameter.'),
    'jawaban_penguji': (
        'Permintaan penguji untuk memperbanyak pengujian dijawab dengan enam eksperimen '
        'tambahan pada notebook ini yang mencakup validasi berulang, estimasi tak bias, '
        'pengujian signifikansi, justifikasi threshold, kalibrasi, dan analisis sensitivitas '
        'hyperparameter; dilengkapi notebook 05 untuk ablation study dan uji robustness.'),
}

hasil_validasi_statistik = {
    'metadata': {
        'notebook'           : '04_Validasi_Statistik_dan_Threshold',
        'mode_cepat'         : bool(MODE_CEPAT),
        'n_data_eksperimen'  : int(len(X_eks)),
        'n_train'            : int(len(X_train)),
        'n_test'             : int(len(X_test)),
        'random_state'       : RANDOM_STATE,
        'alpha'              : ALPHA,
        'threshold_produksi' : THRESHOLD_PRODUKSI,
        'threshold_youden_rf': round(_thr_b, 4),
        'baseline_v2'        : BASELINE_V2,
    },
    'repeated_cv'       : tabel_repeated_cv.to_dict('records'),
    'nested_cv'         : tabel_nested_cv.to_dict('records'),
    'uji_statistik'     : tabel_uji_statistik.to_dict('records'),
    'strategi_threshold': tabel_strategi_threshold.to_dict('records'),
    'kalibrasi'         : tabel_kalibrasi.to_dict('records'),
    'decision_curve'    : tabel_decision_curve.to_dict('records'),
    'validation_curve'  : tabel_validation_curve.to_dict('records'),
    'kesimpulan'        : kesimpulan,
}

simpan_json(hasil_validasi_statistik, 'hasil_validasi_statistik')

garis('DAFTAR BERKAS YANG DIHASILKAN NOTEBOOK 04')
print('TABEL (CSV):')
for t in ['tabel_repeated_cv', 'tabel_nested_cv', 'tabel_uji_statistik',
          'tabel_strategi_threshold', 'tabel_kalibrasi', 'tabel_kalibrasi_per_bin',
          'tabel_decision_curve', 'tabel_validation_curve']:
    print(f'  {OUTPUT_DIR}/tabel/{t}.csv')
print('GAMBAR (PNG):')
for g in ['stat_repeated_cv', 'stat_nested_vs_flat', 'stat_matriks_uji',
          'threshold_strategi', 'stat_kalibrasi', 'stat_decision_curve',
          'stat_validation_curve']:
    print(f'  {OUTPUT_DIR}/gambar/{g}.png')
print('JSON:')
print(f'  {OUTPUT_DIR}/json/hasil_validasi_statistik.json   <- dibaca notebook 06')
print()
print('Notebook 04 selesai.')

---

# RINGKASAN UNTUK SKRIPSI

Bagian ini berisi paragraf siap salin ke naskah skripsi (BAB IV Hasil dan Pembahasan).
Ganti angka di dalam kurung kurawal dengan nilai yang dicetak cell di atas.

---

## 4.x Validasi Statistik Lanjutan

Menanggapi masukan penguji mengenai perlunya memperbanyak pengujian, evaluasi model yang
semula hanya bertumpu pada satu kali pembagian data 80:20 diperluas menjadi enam eksperimen
validasi. Pertama, dilakukan *repeated stratified k-fold cross validation* dengan lima lipatan
dan lima pengulangan sehingga diperoleh 25 estimasi performa untuk setiap model. Kedua,
dilakukan *nested cross validation* dengan lima lipatan luar dan tiga lipatan dalam untuk
memperoleh estimasi performa yang tidak terkontaminasi proses pemilihan hyperparameter.
Ketiga, perbedaan antar model diuji dengan empat uji statistik, yaitu uji McNemar, *5x2cv
paired t-test*, uji Wilcoxon *signed-rank*, dan uji DeLong. Keempat, delapan strategi penentuan
*threshold* dibandingkan secara sistematis. Kelima, kualitas probabilitas keluaran model
dinilai melalui analisis kalibrasi dan *decision curve analysis*. Keenam, sensitivitas model
terhadap hyperparameter diperiksa melalui kurva validasi.

### Cara melaporkan angka (WAJIB dipakai konsisten di seluruh naskah)

Karena setiap eksperimen kini menghasilkan lebih dari satu estimasi, seluruh metrik dilaporkan
dalam bentuk **rata-rata ± margin interval kepercayaan 95%**, bukan angka tunggal:

> Recall Random Forest sebesar **0,9057 ± 0,0083** (interval kepercayaan 95%: 0,8974–0,9140;
> simpangan baku 0,0201; n = 25 lipatan).

Margin dihitung dengan distribusi *t* karena jumlah estimasi tergolong sedikit:

$$\bar{x} \pm t_{0{,}975;\,n-1}\cdot\frac{s}{\sqrt{n}},\qquad n=25,\ df=24,\ t_{0{,}975;24}=2{,}064$$

Aturan penulisan:
1. Tulis rata-rata dan margin dengan **empat angka di belakang koma**.
2. Sertakan **n** (jumlah lipatan) dan **simpangan baku** pada penyebutan pertama di tiap tabel.
3. Untuk hasil uji statistik, laporkan **statistik uji, derajat bebas (bila ada), dan p-value**,
   contoh: *McNemar chi-kuadrat = 12,4507; p = 0,0004*; *5x2cv t(5) = 2,7318; p = 0,0412*;
   *DeLong z = 5,8124; p < 0,001*.
4. Jangan menyimpulkan satu model lebih baik hanya karena rata-ratanya lebih tinggi apabila
   **interval kepercayaan kedua model bertumpang tindih** dan uji statistik menyatakan tidak
   signifikan. Gunakan kalimat "tidak terdapat perbedaan yang signifikan secara statistik".
5. Selalu dampingi p-value dengan ukuran efek (selisih recall atau selisih AUC beserta
   intervalnya), karena pada data berukuran besar p-value mudah menjadi signifikan meskipun
   selisih praktisnya kecil.

### Paragraf hasil (siap salin)

**Validasi berulang.** Hasil *repeated cross validation* menunjukkan Random Forest memperoleh
recall {mean_rf} ± {margin_rf} dan ROC-AUC {auc_rf} ± {margin_auc_rf}, KNN memperoleh recall
{mean_knn} ± {margin_knn}, sedangkan SVM linear memperoleh recall {mean_svm} ± {margin_svm}.
Nilai recall Random Forest yang dilaporkan pada pengujian *holdout* tunggal sebelumnya, yaitu
0,9057, berada di dalam interval kepercayaan 95% hasil validasi berulang sehingga angka
tersebut terbukti bukan hasil kebetulan satu partisi data tertentu.

**Estimasi tak bias.** *Nested cross validation* menghasilkan recall {nested_rf} untuk Random
Forest, lebih rendah {bias_rf} dibanding skor *cross validation* yang diperoleh langsung dari
proses *tuning*. Selisih ini merupakan besarnya bias optimistik yang timbul ketika data
validasi ikut digunakan untuk memilih hyperparameter. Karena besarnya di bawah satu poin
persen, urutan peringkat model tidak berubah dan keputusan pemilihan Random Forest sebagai
model produksi tetap sahih.

**Uji signifikansi.** Empat uji statistik diterapkan pada tiga pasangan model. Uji McNemar
menilai perbedaan pola kesalahan pada data uji yang sama, *5x2cv paired t-test* menilai
perbedaan performa dengan memperhitungkan variasi data latih, uji Wilcoxon menilai perbedaan
tanpa asumsi kenormalan pada 25 skor lipatan, dan uji DeLong menilai perbedaan ROC-AUC dengan
memperhitungkan korelasi antar model karena dievaluasi pada data uji yang identik. Hasil
selengkapnya disajikan pada Tabel 4.x dan Gambar 4.x.

**Justifikasi threshold.** Delapan strategi penentuan *threshold* dibandingkan, yaitu nilai
bawaan 0,5, indeks Youden, F1 maksimum, batas presisi minimal 0,50, batas recall minimal 0,90,
dua skema *cost-sensitive* dengan rasio biaya kesalahan negatif terhadap positif 5:1 dan 10:1,
serta titik terdekat ke koordinat (0,1) pada kurva ROC. Indeks Youden dipilih karena tidak
memerlukan asumsi rasio biaya yang sulit dipertanggungjawabkan, tidak dipengaruhi prevalensi
penyakit, dan menempatkan sistem pada titik kurva ROC terjauh dari garis tebakan acak.
Nilai *threshold* yang dihasilkan, yaitu 0,4965, mendekati 0,5 karena distribusi probabilitas
keluaran model telah diseimbangkan oleh penerapan SMOTE dan pembobotan kelas pada tahap
pelatihan; dengan demikian nilai tersebut merupakan hasil optimasi kriteria Youden, bukan
angka yang ditetapkan secara sembarang.

**Kalibrasi dan utilitas klinis.** *Reliability diagram*, Brier score, dan *Expected
Calibration Error* menunjukkan probabilitas keluaran model layak ditampilkan sebagai persentase
risiko kepada pengguna. *Decision curve analysis* memperlihatkan bahwa pada rentang
*threshold probability* yang relevan untuk skrining awal, penggunaan model memberikan
*net benefit* yang lebih tinggi dibanding strategi merujuk seluruh pasien maupun tidak merujuk
siapa pun, sehingga model terbukti bermanfaat untuk pengambilan keputusan, bukan sekadar
akurat secara statistik.

**Sensitivitas hyperparameter.** Kurva validasi pada parameter jumlah pohon dan kedalaman
maksimum Random Forest, jumlah tetangga KNN, serta parameter regularisasi C pada SVM
memperlihatkan nilai terpilih berada di daerah datar kurva dengan selisih skor latih dan skor
validasi yang kecil. Temuan ini menunjukkan konfigurasi model bersifat stabil dan tidak
mengalami *overfitting*.

---

**Berkas keluaran notebook ini:** delapan tabel CSV, tujuh gambar PNG, dan satu berkas
`hasil_validasi_statistik.json` yang menjadi masukan bagi notebook
`06_Model_Final_dan_Export_Produksi.ipynb`.
